# Анализ фотометрии IC 1613 и Leo A

Диаграммы Г–Р, поиск TRGB, наложение изохрон PARSEC и пространственное распределение.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## 1. Загрузка данных

In [ ]:
import glob

def load_photometry(path, field_name=None):
    df = pd.read_csv(path, sep=r'\s+')
    if field_name is not None:
        df['field'] = field_name
    return df

ic_files = sorted(glob.glob('photometry_IC1613/IC1613*.dat'))

parts = []
for path in ic_files:
    fname = path.split('/')[-1]
    field = fname.replace('IC1613', '').replace('.dat', '')
    parts.append(load_photometry(path, field_name=field))

df_ic1613 = pd.concat(parts, ignore_index=True)

df_leoa = load_photometry('photometry_LeoA/LeoA.phot.dat', field_name='LeoA')

print(f"IC 1613 : {len(df_ic1613):,} звёзd | поля: {df_ic1613['field'].value_counts().to_dict()}")
print(f"Leo A   : {len(df_leoa):,} звёзd")



## 2. Диаграмма Г–Р (CMD) — обе галактики

In [ ]:
color_ic1613     = df_ic1613['mag_F475W'] - df_ic1613['mag_F814W']
magnitude_ic1613 = df_ic1613['mag_F814W']

color_leoa       = df_leoa['mag_F475W'] - df_leoa['mag_F814W']
magnitude_leoa   = df_leoa['mag_F814W']

fig, axes = plt.subplots(2, 1, figsize=(10, 14))
for ax, color, mag, title in zip(
        axes,
        [color_ic1613, color_leoa],
        [magnitude_ic1613, magnitude_leoa],
        ['CMD IC 1613', 'CMD Leo A']):
    ax.scatter(color, mag, s=1, alpha=0.2, c='black')
    ax.invert_yaxis()
    ax.set_xlabel('mag_F475W - mag_F814W')
    ax.set_ylabel('mag_F814W')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Выделение ветви красных гигантов (RGB-фильтрация)

In [ ]:
# IC 1613  —  диапазоны подобраны вручную (как в smth_Илья.ipynb)
mask_ic1613 = (
    (color_ic1613 > 1.9) & (color_ic1613 < 3.0) &
    (magnitude_ic1613 > 20.0) & (magnitude_ic1613 < 22.5)
)
filtered_color_ic1613 = color_ic1613[mask_ic1613]
filtered_mag_ic1613   = magnitude_ic1613[mask_ic1613]

# Leo A  —  не захватываем сверхгигантов
mask_leoa = (
    (color_leoa > 1.3) & (color_leoa < 2.25) &
    (magnitude_leoa > 20.0) & (magnitude_leoa < 23.5)
)
filtered_color_leoa = color_leoa[mask_leoa]
filtered_mag_leoa   = magnitude_leoa[mask_leoa]

fig, axes = plt.subplots(2, 1, figsize=(10, 14))
for ax, c_all, m_all, c_sel, m_sel, title in zip(
        axes,
        [color_ic1613, color_leoa],
        [magnitude_ic1613, magnitude_leoa],
        [filtered_color_ic1613, filtered_color_leoa],
        [filtered_mag_ic1613, filtered_mag_leoa],
        ['IC 1613', 'Leo A']):
    ax.scatter(c_all, m_all, s=1, alpha=0.1, c='black', label='All')
    ax.scatter(c_sel, m_sel, s=2, alpha=0.5, c='red',   label='RGB selection')
    ax.invert_yaxis()
    ax.set_xlabel('F475W - F814W'); ax.set_ylabel('F814W')
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Функция светимости + фильтр Собеля → TRGB

In [ ]:
def gaussian_kernel_1d(sigma, radius=None):
    if radius is None:
        radius = int(3 * sigma)
    x = np.arange(-radius, radius + 1)
    kernel = np.exp(-0.5 * (x / sigma) ** 2)
    return kernel / kernel.sum()

sobel_kernel = np.array([1, 0, -1])
gauss_kernel = gaussian_kernel_1d(sigma=1.5)

def get_trgb(filtered_mag, bins=50):
    counts, edges = np.histogram(filtered_mag, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    smooth  = np.convolve(counts, gauss_kernel, mode='same')
    sobel   = np.convolve(smooth, sobel_kernel,  mode='same')
    sobel_raw = np.convolve(counts, sobel_kernel, mode='same')
    idx = np.where(
        (sobel[1:-1] > sobel[:-2]) & (sobel[1:-1] > sobel[2:])
    )[0]
    trgb_mag = centers[idx[0] + 1] if len(idx) else None
    return counts, edges, centers, smooth, sobel, sobel_raw, trgb_mag

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for row, (filtered_mag, title) in enumerate(
        [(filtered_mag_ic1613, 'IC 1613'), (filtered_mag_leoa, 'Leo A')]):
    counts, edges, centers, smooth, sobel, sobel_raw, trgb = get_trgb(filtered_mag)
    axes[row, 0].hist(filtered_mag, bins=50, color='lightgray', edgecolor='black')
    if trgb: axes[row, 0].axvline(trgb, color='blue', lw=2, label=f'TRGB ≈ {trgb:.2f}')
    axes[row, 0].set_title(f'Luminosity Function: {title}')
    axes[row, 0].set_xlabel('F814W'); axes[row, 0].legend(); axes[row, 0].grid(True, alpha=0.3)
    axes[row, 1].plot(centers, sobel_raw, color='gray', ls='--', alpha=0.6, label='Sobel (raw)')
    axes[row, 1].plot(centers, sobel,     color='red',              label='Sobel (smoothed)')
    if title == 'IC 1613':
        axes[row, 1].set_ylim(-45, 55)   # больше звёзд → выше пик Sobel
    else:
        axes[row, 1].set_ylim(-20, 25)
    if trgb: axes[row, 1].axvline(trgb, color='blue', lw=2, label=f'TRGB ≈ {trgb:.2f}')
    axes[row, 1].set_title(f'Sobel Response: {title}')
    axes[row, 1].set_xlabel('F814W'); axes[row, 1].legend(); axes[row, 1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

_, _, _, _, _, _, trgb_ic1613 = get_trgb(filtered_mag_ic1613)
_, _, _, _, _, _, trgb_leoa   = get_trgb(filtered_mag_leoa)
print(f"TRGB IC 1613 : {trgb_ic1613:.3f} mag")
print(f"TRGB Leo A   : {trgb_leoa:.3f} mag")

## 5. Вычисление расстояний

In [ ]:
M_TRGB = -4.05          # абсолютная звёздная величина TRGB в F814W

# поглощения на полосу F814W
A_ic1613 = 0.038
A_leoa = 0.032

def distance_kpc(m_trgb, A):
    return round(10 ** ((m_trgb - A - M_TRGB + 5) / 5) / 1000)

dist_ic1613 = distance_kpc(trgb_ic1613, A_ic1613)
dist_leoa   = distance_kpc(trgb_leoa,   A_leoa)

print(f"Расстояние до IC 1613 : {dist_ic1613} кпк  (лит. 730 ± 20 кпк)")
print(f"Расстояние до Leo A   : {dist_leoa} кпк  (лит. 790 ± 40 кпк)")

## 6. Matplotlib: наложение размеченных изохрон PARSEC

Файлы `iso1.dat` (IC 1613) и `iso2.dat` (Leo A) — изохроны PARSEC с колонкой `label` (стадии эволюции).

In [ ]:
A_B_ic1613 = 0.082
A_B_leoa   = 0.068

parsec_descriptions = {
    0: 'PMS', 1: 'MS', 2: 'SGB', 3: 'RGB',
    4: 'CHEB', 5: 'CHEB', 6: 'CHEB',
    7: 'EAGB', 8: 'TPAGB'
}
color_map = {
    0: 'blue', 1: 'green', 2: 'yellow', 3: 'orange',
    4: 'red',  5: 'red',   6: 'red',
    7: 'purple', 8: 'brown'
}

def load_labeled_iso(path, dist_kpc, A_B, A_I):
    columns = None
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('# Zini'):
                columns = line.strip().lstrip('#').split()
                break
    df = pd.read_csv(path, sep=r'\s+', comment='#', names=columns)
    df = df[df['label'] != 9].copy()
    mu = 5 * np.log10(dist_kpc * 1000) - 5
    df['F814Wmag_app'] = df['F814Wmag'] + mu + A_I
    df['F475Wmag_app'] = df['F475Wmag'] + mu + A_B
    df['label_merged'] = df['label'].replace({5: 4, 6: 4})
    return df

def plot_with_labeled_iso(ax, obs_color, obs_mag, iso_df, title):
    ax.scatter(obs_color, obs_mag, s=1, alpha=0.8, c='black', zorder=1)
    unique_labels = np.unique(iso_df['label_merged'])
    for lbl in unique_labels:
        m = iso_df['label_merged'] == lbl
        ax.scatter(
            iso_df.loc[m, 'F475Wmag_app'] - iso_df.loc[m, 'F814Wmag_app'],
            iso_df.loc[m, 'F814Wmag_app'],
            s=10, alpha=0.4, color=color_map.get(lbl, 'black'),
            label=parsec_descriptions.get(lbl, str(lbl))
        )
    ax.invert_yaxis()
    ax.set_xlim(-1, 3); ax.set_ylim(28, 18)
    ax.set_xlabel('F475W - F814W'); ax.set_ylabel('F814W')
    ax.set_title(title); ax.legend(title='Стадия', fontsize=8); ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

import os
# IC 1613
if os.path.exists('iso1.dat'):
    iso_ic = load_labeled_iso('iso1.dat', dist_ic1613, A_B_ic1613, A_ic1613)
    plot_with_labeled_iso(axes[0], color_ic1613, magnitude_ic1613, iso_ic, 'IC 1613 + Isochrones (PARSEC)')
else:
    axes[0].set_title('Файл iso1.dat не найден')

if os.path.exists('iso2.dat'):
    iso_la = load_labeled_iso('iso2.dat', dist_leoa, A_B_leoa, A_leoa)
    plot_with_labeled_iso(axes[1], color_leoa, magnitude_leoa, iso_la, 'Leo A + Isochrones (PARSEC)')
else:
    axes[1].set_title('Файл iso2.dat не найден')

plt.tight_layout(); plt.show()

---
## 7. Plotly: интерактивное наложение всей сетки изохрон



In [ ]:
A_B_ic1613 = 0.082
A_B_leoa   = 0.068

parsec_descriptions = {
    0: 'PMS', 1: 'MS', 2: 'SGB', 3: 'RGB',
    4: 'CHEB', 5: 'CHEB', 6: 'CHEB',
    7: 'EAGB', 8: 'TPAGB'
}
color_map = {
    0: 'blue', 1: 'green', 2: 'yellow', 3: 'orange',
    4: 'red',  5: 'red',   6: 'red',
    7: 'purple', 8: 'brown'
}


import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

parsec_abbr = {
    0: 'PMS', 1: 'MS', 2: 'SGB', 3: 'RGB',
    4: 'CHEB1', 5: 'CHEB2', 6: 'CHEB3',
    7: 'EAGB', 8: 'TPAGB', 9: 'post-AGB'
}

def load_isochrones(path):
    """Читает файл изохрон PARSEC, определяя колонки из заголовка # Zini ...""";
    columns = None
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('# Zini'):
                columns = line.strip().lstrip('#').split()
                break
    df = pd.read_csv(path, sep=r'\s+', comment='#', names=columns)
    df = df[df['label'] != 9].copy()          # убираем post-AGB
    df['stage']     = df['label'].map(parsec_abbr)
    df['color_mag'] = df['F475Wmag'] - df['F814Wmag']
    return df

# Конвертация наблюдений в абсолютные величины
def to_absolute(app_mag, dist_kpc, A):
    mu = 5 * np.log10(dist_kpc * 1000) - 5
    return app_mag - mu - A

mag_F475W_abs_ic1613 = to_absolute(df_ic1613['mag_F475W'], dist_ic1613, A_B_ic1613)
mag_F814W_abs_ic1613 = to_absolute(magnitude_ic1613,   dist_ic1613, A_ic1613)
color_abs_ic1613     = mag_F475W_abs_ic1613 - mag_F814W_abs_ic1613

mag_F475W_abs_leoa   = to_absolute(df_leoa['mag_F475W'], dist_leoa, A_B_leoa)
mag_F814W_abs_leoa   = to_absolute(magnitude_leoa,       dist_leoa, A_leoa)
color_abs_leoa       = mag_F475W_abs_leoa - mag_F814W_abs_leoa

import os
from pathlib import Path

def _find_iso(*names):
    roots = [Path.cwd().resolve(), Path('/home/olga/SAO/conference'), Path('/home/olga/SAO/TRGB/files')]
    roots.extend(Path.cwd().resolve().parents)
    seen = set()
    for root in roots:
        root = Path(root)
        if root in seen:
            continue
        seen.add(root)
        for name in names:
            cand = root / name
            if cand.is_file():
                return cand
    return None

iso_path = _find_iso('iso_sum.dat', 'iso3.dat')
print("Загружаем изохроны...")
if iso_path is None:
    iso_sum = None
    print("Файл изохрон не найден: iso_sum.dat или iso3.dat")
else:
    iso_sum = load_isochrones(iso_path)
    print(f"{iso_path.name}: {len(iso_sum):,} точек | MH: {sorted(iso_sum['MH'].unique())} | logAge: {len(iso_sum['logAge'].unique())} значений")
    print(iso_path)


### 7a. Plotly CMD со всеми изохронами сразу (прокрутка колёсиком мыши)

In [ ]:
def make_overlay_fig(color_abs, mag_abs, galaxy_name, iso_df, xlim=(-1, 4), ylim=(-6, 8)):
    """Plotly-фигура: наблюдения + изохроны, раскрашенные по стадии.""";
    traces = []
    traces.append(go.Scattergl(
        x=color_abs, y=mag_abs, mode='markers',
        marker=dict(color='black', size=2, opacity=0.3),
        name=galaxy_name,
        hovertemplate='color=%{x:.2f}<br>mag=%{y:.2f}<extra></extra>'
    ))
    if iso_df is not None:
        iso_plot = iso_df[
            (iso_df['color_mag'] >= xlim[0]) & (iso_df['color_mag'] <= xlim[1]) &
            (iso_df['F814Wmag'] >= ylim[0]) & (iso_df['F814Wmag'] <= ylim[1])
        ]
        palette = px.colors.qualitative.Plotly
        for i, (stage, grp) in enumerate(iso_plot.groupby('stage')):
            if pd.isna(stage):
                continue
            traces.append(go.Scattergl(
                x=grp['color_mag'], y=grp['F814Wmag'], mode='markers',
                marker=dict(size=4, opacity=0.7, color=palette[i % len(palette)]),
                name=str(stage),
                hovertemplate=f'{stage}<br>color=%{{x:.2f}}<br>mag=%{{y:.2f}}<extra></extra>'
            ))
    fig = go.Figure(traces)
    fig.update_layout(
        xaxis=dict(title='M_F475W − M_F814W', range=list(xlim)),
        yaxis=dict(title='M_F814W', range=list(ylim), autorange='reversed'),
        title=f'{galaxy_name} + все изохроны',
        width=750, height=620,
        dragmode='pan'
    )
    return fig

if iso_sum is not None:
    fig_ic = make_overlay_fig(color_abs_ic1613, mag_F814W_abs_ic1613, 'IC1613', iso_sum)
    fig_ic.show(config={'scrollZoom': True})

In [ ]:
if iso_sum is not None:
    fig_la = make_overlay_fig(color_abs_leoa, mag_F814W_abs_leoa, 'LeoA', iso_sum)
    fig_la.show(config={'scrollZoom': True})

### 7b. 🔲 Интерактивная сетка чекбоксов MH × возраст (Гyr)

Включайте/выключайте отдельные изохроны прямо на графике.

In [ ]:
def make_toggle_widget(color_abs, mag_abs, galaxy_name, iso_df):
    """
    Plotly CMD + таблица чекбоксов:
    строки = металличность ([M/H], Z),
    столбцы = возраст (logAge и обычная шкала).
    По умолчанию изохроны скрыты.
    """
    if iso_df is None:
        print("Изохроны не загружены.")
        return

    mh_values  = sorted(iso_df['MH'].unique())
    age_values = sorted(iso_df['logAge'].unique())

    # Для каждой [M/H] берем соответствующий Zini из PARSEC
    zini_map = {}
    if 'Zini' in iso_df.columns:
        for mh in mh_values:
            zini_map[mh] = iso_df.loc[np.isclose(iso_df['MH'], mh), 'Zini'].iloc[0]

    traces = [go.Scattergl(
        x=color_abs,
        y=mag_abs,
        mode='markers',
        marker=dict(color='black', size=2, opacity=0.3),
        name=galaxy_name
    )]

    trace_index = {}
    palette = px.colors.qualitative.Plotly

    for i, (mh, age) in enumerate((mh, age) for mh in mh_values for age in age_values):
        grp = iso_df[
            (np.isclose(iso_df['MH'], mh)) &
            (np.isclose(iso_df['logAge'], age))
        ]

        if grp.empty:
            continue

        idx = len(traces)
        trace_index[(mh, age)] = idx

        traces.append(go.Scatter(
            x=grp['color_mag'],
            y=grp['F814Wmag'],
            mode='markers',
            marker=dict(size=5, color=palette[i % len(palette)], opacity=0.8),
            name=f'[M/H]={mh:.1f}, Z={zini_map.get(mh, np.nan):.2e}, logAge={age:.2f}',
            visible=False
        ))

    fig_widget = go.FigureWidget(traces)
    fig_widget.update_layout(
        yaxis_autorange='reversed',
        xaxis_title='M_F475W − M_F814W',
        yaxis_title='M_F814W',
        title=f'{galaxy_name}: включите нужные изохроны в таблице ниже',
        width=760,
        height=580,
        dragmode='pan'
    )

    mh_width = '72px'
    z_width = '100px'
    age_width = '64px'

    def fixed(width):
        return widgets.Layout(width=width, min_width=width)

    def centered(w, width=age_width):
        return widgets.HBox(
            [w],
            layout=widgets.Layout(
                justify_content='center',
                width=width,
                min_width=width
            )
        )

    def bold(text, width=None):
        return widgets.HTML(
            f'<b>{text}</b>',
            layout=fixed(width) if width else widgets.Layout()
        )

    def format_age(log_age):
        age_yr = 10 ** log_age
        if age_yr < 1e9:
            return f'{age_yr / 1e6:.0f} Myr'
        return f'{age_yr / 1e9:.2g} Gyr'

    rows = []

    # Верхняя строка: logAge
    rows.append(widgets.HBox(
        [bold('[M/H]', mh_width), bold('Z', z_width)] +
        [centered(bold(f'{age:.2f}')) for age in age_values]
    ))

    # Вторая строка: обычный возраст
    rows.append(widgets.HBox(
        [
            widgets.Label('', layout=fixed(mh_width)),
            widgets.Label('', layout=fixed(z_width))
        ] +
        [centered(widgets.Label(format_age(age))) for age in age_values]
    ))

    checkboxes = {}

    for mh in mh_values:
        row = [
            widgets.Label(f'{mh:.2f}', layout=fixed(mh_width)),
            widgets.Label(
                f'{zini_map[mh]:.2e}' if mh in zini_map else '—',
                layout=fixed(z_width)
            )
        ]

        for age in age_values:
            cb = widgets.Checkbox(
                value=False,
                indent=False,
                layout=widgets.Layout(width='35px')
            )

            if (mh, age) in trace_index:
                def on_change(change, mh=mh, age=age):
                    fig_widget.data[trace_index[(mh, age)]].visible = change['new']

                cb.observe(on_change, names='value')
            else:
                cb.disabled = True

            checkboxes[(mh, age)] = cb
            row.append(centered(cb))

        rows.append(widgets.HBox(row))

    def set_all(value):
        with fig_widget.batch_update():
            for (mh, age), cb in checkboxes.items():
                if cb.disabled:
                    continue

                cb.value = value
                fig_widget.data[trace_index[(mh, age)]].visible = value

    btn_clear = widgets.Button(description='Убрать все', button_style='warning')
    btn_fill = widgets.Button(description='Заполнить все', button_style='success')

    btn_clear.on_click(lambda _: set_all(False))
    btn_fill.on_click(lambda _: set_all(True))

    controls = widgets.HBox(
        [btn_clear, btn_fill],
        layout=widgets.Layout(
            justify_content='center',
            margin='8px 0 8px 0'
        )
    )

    header_note = widgets.HTML(
        '<b>Таблица изохрон:</b> строки — металличность ([M/H], Z), '
        'столбцы — возраст (logAge сверху, обычный возраст ниже).'
    )

    grid = widgets.VBox(rows, layout=widgets.Layout(align_items='flex-start'))

    return widgets.VBox(
    [fig_widget, controls, header_note, grid],
    layout=widgets.Layout(align_items='flex-start')
)


if iso_sum is not None:
    print("IC 1613:")
    display(make_toggle_widget(
        color_abs_ic1613,
        mag_F814W_abs_ic1613,
        'IC1613',
        iso_sum
    ))

In [ ]:
if iso_sum is not None:
    print("Leo A:")
    display(make_toggle_widget(color_abs_leoa, mag_F814W_abs_leoa, 'LeoA', iso_sum))

## 8. Пространственная карта IC 1613 (RA, Dec)

Сначала все шесть полей в общих координатах, каждое своим цветом.
Затем параллелограмм каждого поля и приоритет: в перекрытии остаётся одно поле.
Дальше по тексту рабочая выборка — только звёзды **внутри** параллелограмма своего поля (`df_ic1613_in`). Бахрома за контуром отбрасывается.
Стадия звезды — ближайшая точка изохроны на диаграмме цвет–величина.


In [ ]:
# Все 6 полей IC 1613 в общих координатах (RA, Dec).
# Цвет = файл; легенда — имя файла.
fig, ax = plt.subplots(figsize=(10, 8))
_colors = plt.cm.tab10.colors
_fields = sorted(df_ic1613['field'].unique())

for _i, _field in enumerate(_fields):
    _s = df_ic1613[df_ic1613['field'] == _field]
    _fname = f'IC1613{_field}.dat'
    ax.scatter(
        _s['ra'], _s['dec'],
        s=0.4, alpha=0.35, c=[_colors[_i % len(_colors)]],
        label=f'{_fname}  ({len(_s):,})',
        rasterized=True,
    )

ax.set_xlabel('α = RA (deg)')
ax.set_ylabel('δ = Dec (deg)')

ax.set_title('IC 1613: все поля HST')
ax.invert_xaxis()
ax.legend(markerscale=10, fontsize=9, loc='best', framealpha=0.95)
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
plt.show()


In [ ]:
def classify_by_isochrone(obs_color, obs_mag, iso_df, chunk_size=5000):
    """
    Для каждой наблюдаемой звезды находит ближайшую точку изохроны
    и присваивает ей соответствующую стадию эволюции.
    Обработка чанками, чтобы не переполнить память.
    """
    if iso_df is None or len(iso_df) == 0:
        raise ValueError('iso_df пуст — для выбранного MH нет точек. Доступные MH: -2.0 … -0.6 (шаг 0.2)')
    iso_xy    = np.column_stack([iso_df['color_mag'].to_numpy(),
                                  iso_df['F814Wmag'].to_numpy()])
    iso_label = iso_df['label'].to_numpy()
    obs_color = np.asarray(obs_color, dtype=float)
    obs_mag   = np.asarray(obs_mag,   dtype=float)
    obs_xy    = np.column_stack([obs_color, obs_mag])

    result = np.empty(len(obs_xy), dtype=int)
    iso_sq = (iso_xy ** 2).sum(axis=1)

    for start in range(0, len(obs_xy), chunk_size):
        end  = min(start + chunk_size, len(obs_xy))
        chunk = obs_xy[start:end]
        # квадрат евклидова расстояния через матричное умножение
        dists = iso_sq - 2 * chunk @ iso_xy.T + (chunk ** 2).sum(axis=1, keepdims=True)
        result[start:end] = iso_label[np.argmin(dists, axis=1)]

    return result

In [ ]:
def make_spatial_widget(coord_x, coord_y, labels, galaxy_name,
                        x_label='α = RA (deg)', y_label='δ = Dec (deg)',
                        reverse_x=True):
    """
    Plotly-карта пространственного распределения.

    Для IC 1613 используем небесные координаты α/δ (ra/dec),
    чтобы корректно наложить все 6 полей друг на друга.
    Для объектов без ra/dec можно передавать обычные x/y и reverse_x=False.

    Крестом отмечается эмпирический центр: максимум 2D-плотности звезд.
    """
    coord_x = np.asarray(coord_x, dtype=float)
    coord_y = np.asarray(coord_y, dtype=float)
    labels = np.asarray(labels)

    finite = np.isfinite(coord_x) & np.isfinite(coord_y)
    coord_x = coord_x[finite]
    coord_y = coord_y[finite]
    labels = labels[finite]

    # Центр по максимуму плотности звезд на общей координатной сетке.
    H, xedges, yedges = np.histogram2d(coord_x, coord_y, bins=100)
    idx = np.unravel_index(np.argmax(H), H.shape)
    center_x = (xedges[idx[0]] + xedges[idx[0] + 1]) / 2
    center_y = (yedges[idx[1]] + yedges[idx[1] + 1]) / 2

    unique_labels = np.unique(labels)
    palette = px.colors.qualitative.Plotly
    traces = []
    trace_index = {}

    for i, lbl in enumerate(unique_labels):
        m = labels == lbl
        stage = parsec_abbr.get(int(lbl), f'Stage {lbl}')
        traces.append(go.Scattergl(
            x=coord_x[m],
            y=coord_y[m],
            mode='markers',
            marker=dict(size=2, color=palette[i % len(palette)], opacity=0.6),
            name=stage
        ))
        trace_index[lbl] = len(traces) - 1

    else:
        traces.append(go.Scattergl(
            x=[center_x],
            y=[center_y],
            mode='markers',
            marker=dict(size=15, color='black', symbol='cross', line=dict(width=2, color='white')),
            name='Density peak center'
        ))

    fig_widget = go.FigureWidget(traces)
    fig_widget.update_layout(
        xaxis_title=x_label,
        yaxis_title=y_label,
        title=f'Spatial Distribution: {galaxy_name}',
        width=760,
        height=650,
        dragmode='pan'
    )

    if reverse_x:
        fig_widget.update_xaxes(autorange='reversed')

    checkboxes = []
    for lbl in unique_labels:
        stage = parsec_abbr.get(int(lbl), f'Stage {lbl}')
        cb = widgets.Checkbox(
            value=True,
            description=stage,
            indent=False,
            layout=widgets.Layout(width='90px')
        )

        def on_toggle(change, lbl=lbl):
            fig_widget.data[trace_index[lbl]].visible = change['new']

        cb.observe(on_toggle, names='value')
        checkboxes.append(cb)

    controls = widgets.VBox(checkboxes)
    return widgets.HBox([fig_widget, controls])

# ═══════════════════════════════════════════════════════════════════════════
# IC 1613: границы полей — параллелограммы по облаку (RA, Dec).
#
# Для каждого поля: ConvexHull → два доминирующих направления сторон →
# нормали → 2-й и 98-й перцентили проекций. Четыре прямые Ax + By = C.
# Точка внутри поля ⟺ она между каждой парой параллельных прямых.
#
# Приоритет 1→6: звезда пропускается, если её (RA, Dec) уже внутри
# более приоритетного параллелограмма.
#
# df_ic1613       — полный каталог
# df_ic1613_nodup — без наложений, для пространственной карты
# ═══════════════════════════════════════════════════════════════════════════

IC1613_FIELD_PRIORITY = [
    'F33_34',   # 1
    'F27_28',   # 2
    'F19_ACS',  # 3
    'F17_20',   # 4
    'F11_14',   # 5
    'F05_08',   # 6
]


def fit_parallelogram_lines(x, y):
    """Четыре прямые Ax + By = C, образующие параллелограмм вокруг облака точек.

    1. ConvexHull.
    2. Углы рёбер оболочки в [0, π).
    3. Два доминирующих направления (гистограмма углов с весом = длина ребра).
    4. Нормали к этим направлениям.
    5. Проекция всех точек на нормали; 2-й и 98-й перцентили — отступы прямых.
    """
    from scipy.spatial import ConvexHull

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    pts = np.column_stack([x, y])
    hull = ConvexHull(pts)
    hv = pts[hull.vertices]
    edges = np.roll(hv, -1, axis=0) - hv
    lengths = np.hypot(edges[:, 0], edges[:, 1])
    angles = np.arctan2(edges[:, 1], edges[:, 0]) % np.pi

    n_bins = 180
    hist, _ = np.histogram(angles, bins=n_bins, range=(0.0, np.pi), weights=lengths)
    kernel = np.array([1.0, 2.0, 3.0, 2.0, 1.0])
    kernel /= kernel.sum()
    smooth = np.convolve(np.r_[hist[-2:], hist, hist[:2]], kernel, mode='valid')
    centers = (np.arange(n_bins) + 0.5) * (np.pi / n_bins)

    i1 = int(np.argmax(smooth))
    ang1 = float(centers[i1])
    dist = np.abs(centers - ang1)
    dist = np.minimum(dist, np.pi - dist)
    smooth2 = smooth.copy()
    smooth2[dist < np.deg2rad(20)] = -1.0
    ang2 = float(centers[int(np.argmax(smooth2))])

    lines = []
    for ang in (ang1, ang2):
        normal = np.array([-np.sin(ang), np.cos(ang)])
        proj = pts @ normal
        c_lo, c_hi = np.percentile(proj, [2, 98])
        lines.append([normal[0], normal[1], c_lo])
        lines.append([normal[0], normal[1], c_hi])
    return np.asarray(lines, dtype=float)


def in_field_parallelogram(ra, dec, lines):
    """Точка внутри параллелограмма: между каждой парой параллельных прямых."""
    ra = np.asarray(ra, dtype=float)
    dec = np.asarray(dec, dtype=float)
    p0 = lines[0, 0] * ra + lines[0, 1] * dec
    p1 = lines[2, 0] * ra + lines[2, 1] * dec
    c0_lo, c0_hi = np.sort(lines[0:2, 2])
    c1_lo, c1_hi = np.sort(lines[2:4, 2])
    return (p0 >= c0_lo) & (p0 <= c0_hi) & (p1 >= c1_lo) & (p1 <= c1_hi)


def parallelogram_corners(lines):
    """Вершины = пересечения четырёх прямых, по порядку обхода."""
    n1 = lines[0, :2]
    n2 = lines[2, :2]
    M = np.vstack([n1, n2])
    corners = []
    for c1 in lines[0:2, 2]:
        for c2 in lines[2:4, 2]:
            corners.append(np.linalg.solve(M, [c1, c2]))
    corners = np.asarray(corners)
    center = corners.mean(axis=0)
    order = np.arctan2(corners[:, 1] - center[1], corners[:, 0] - center[0]).argsort()
    corners = corners[order]
    corners = np.vstack([corners, corners[0]])
    return corners[:, 0], corners[:, 1]


ic1613_lines = {
    _field: fit_parallelogram_lines(
        df_ic1613.loc[df_ic1613['field'] == _field, 'ra'],
        df_ic1613.loc[df_ic1613['field'] == _field, 'dec'],
    )
    for _field in IC1613_FIELD_PRIORITY
}

_ra = df_ic1613['ra'].to_numpy()
_dec = df_ic1613['dec'].to_numpy()
_keep_spatial = np.zeros(len(df_ic1613), dtype=bool)

for _i, _field in enumerate(IC1613_FIELD_PRIORITY):
    _claimed = np.zeros(len(df_ic1613), dtype=bool)
    for _prev in IC1613_FIELD_PRIORITY[:_i]:
        _claimed |= in_field_parallelogram(_ra, _dec, ic1613_lines[_prev])
    _take = (df_ic1613['field'].to_numpy() == _field) & ~_claimed
    _keep_spatial |= _take
    print(f"  [{_i + 1}] {_field}: +{_take.sum():,}")

df_ic1613_nodup = df_ic1613.loc[_keep_spatial].reset_index(drop=True)
print(f"\nПолный каталог df_ic1613:         {len(df_ic1613):,}")
print(f"Без наложений df_ic1613_nodup:    {len(df_ic1613_nodup):,}")
print(f"Пропущено (внутри более приоритетного поля): {(~_keep_spatial).sum():,}")

fig, ax = plt.subplots(figsize=(9, 7))
for _i, (_field, _c) in enumerate(zip(IC1613_FIELD_PRIORITY, plt.cm.tab10.colors)):
    _s = df_ic1613_nodup[df_ic1613_nodup['field'] == _field]
    ax.scatter(_s['ra'], _s['dec'], s=0.15, alpha=0.25, c=[_c],
               label=f'{_i + 1}: IC1613{_field}.dat', rasterized=True)
    _rx, _ry = parallelogram_corners(ic1613_lines[_field])
    ax.plot(_rx, _ry, color=_c, lw=1.4)

ax.set_xlabel('α = RA (deg)')
ax.set_ylabel('δ = Dec (deg)')
ax.set_title('IC 1613: параллелограммы (ConvexHull, 2–98 перцентили)')
ax.invert_xaxis()
ax.set_aspect('equal', adjustable='datalim')
ax.legend(markerscale=8, fontsize=8, loc='best')
plt.tight_layout()
plt.show()

# Классификация — по полному каталогу; на карту берём подмножество
if iso_sum is not None:
    print("Классификация IC 1613 (полный каталог)...")
    label_ic1613 = classify_by_isochrone(
        color_abs_ic1613, mag_F814W_abs_ic1613,
        iso_sum[np.isclose(iso_sum['MH'], -1.0)]
    )
    label_ic1613_nodup = label_ic1613[_keep_spatial]

    # рабочая выборка с пункта 8: внутри параллелограмма своего поля
    _inside = np.zeros(len(df_ic1613_nodup), dtype=bool)
    _fields = df_ic1613_nodup['field'].to_numpy()
    _ra_nd = df_ic1613_nodup['ra'].to_numpy()
    _dec_nd = df_ic1613_nodup['dec'].to_numpy()
    for _field, _lines in ic1613_lines.items():
        _m = _fields == _field
        _inside[_m] = in_field_parallelogram(_ra_nd[_m], _dec_nd[_m], _lines)
    df_ic1613_in = df_ic1613_nodup.loc[_inside].reset_index(drop=True)
    label_ic1613_in = label_ic1613_nodup[_inside]
    print(f"Внутри параллелограммов df_ic1613_in: {len(df_ic1613_in):,} "
          f"(отброшено {(~_inside).sum():,} за контуром)")

    print("Классификация Leo A...")
    label_leoa = classify_by_isochrone(
        color_abs_leoa, mag_F814W_abs_leoa,
        iso_sum[np.isclose(iso_sum['MH'], -1.4)]
    )
    print("Готово!")


In [ ]:
if iso_sum is not None:
    print("IC 1613 — пространственное распределение по стадиям:")
    display(make_spatial_widget(
        df_ic1613_in['ra'].to_numpy(), df_ic1613_in['dec'].to_numpy(),
        label_ic1613_in, 'IC1613',
        x_label='α = RA (deg)', y_label='δ = Dec (deg)', reverse_x=True
    ))


In [ ]:
if iso_sum is not None:
    print("Leo A — пространственное распределение по стадиям:")
    display(make_spatial_widget(
        df_leoa['x'].to_numpy(), df_leoa['y'].to_numpy(),
        label_leoa, 'LeoA',
        x_label='X (pixels)', y_label='Y (pixels)', reverse_x=False
    ))

---
## 9. Координаты

У IC 1613 в каталогах уже есть `ra` и `dec` (J2000 / ICRS). Пиксели `x`, `y` локальны для каждого поля и между полями не складываются.
С пункта 8 для карт и профилей IC 1613 используется `df_ic1613_in`: без повторов полей и только внутри параллелограмма своего кадра.


In [ ]:
# ПРИМЕР КОДА (раскомментируйте, если есть FITS-файл)

# from astropy.wcs import WCS
# from astropy.io import fits
# from astropy.coordinates import SkyCoord
# import astropy.units as u

# 1. Читаем заголовок снимка
# hdu = fits.open('IC1613_reference_image.fits')
# wcs = WCS(hdu[0].header)

# 2. Задаем координаты центра из NED (например, для IC 1613)
# center_coord = SkyCoord('01h04m47.8s', '+02d07m04s', frame='icrs')

# 3. Переводим RA/Dec в пиксели X/Y
# center_x, center_y = wcs.world_to_pixel(center_coord)
# print(f"Центр в пикселях: X={center_x}, Y={center_y}")

# 4. Перевод всего каталога звезд из X/Y в RA/Dec (если нужно)
# ra, dec = wcs.pixel_to_world_values(df_ic1613['x'], df_ic1613['y'])
# df_ic1613['RA'] = ra
# df_ic1613['Dec'] = dec

---
## 10. Радиальное распределение звезд (Leo A)

Считаем расстояние каждой звезды от эмпирического центра и строим профиль плотности (концентрации) для разных стадий эволюции. Это позволяет наглядно увидеть, что молодые звезды (MS, CHEB) сильнее сконцентрированы к центру, а старые (RGB) образуют протяженное гало.

In [ ]:
# ЧТО ЗДЕСЬ ПРОИСХОДИТ:
# 1. Мы берем координаты X и Y всех звезд Leo A.
# 2. Строим 2D-гистограмму, чтобы найти самую густонаселенную точку (это будет наш эмпирический центр cx, cy).
# 3. Для каждой звезды вычисляем ее расстояние R от этого центра по теореме Пифагора.
# 4. Разбиваем галактику на кольца (от 0 до 1800 пикселей) и считаем площадь каждого кольца.
# 5. Для каждой стадии эволюции (RGB, MS, CHEB и т.д.) считаем, сколько звезд попало в каждое кольцо.
# 6. Делим количество звезд на площадь кольца — получаем истинную поверхностную плотность (концентрацию).
# 7. Строим два графика: абсолютную плотность и нормированную (где плотность в центре = 1 для всех стадий).
# Вывод: молодые звезды (MS, CHEB) резко падают при удалении от центра, а старые (RGB) образуют широкое гало.

import numpy as np
import matplotlib.pyplot as plt

# 1. Эмпирический центр Leo A (по пику плотности)
x_la = df_leoa['x'].to_numpy()
y_la = df_leoa['y'].to_numpy()

H, xedges, yedges = np.histogram2d(x_la, y_la, bins=100)
idx = np.unravel_index(np.argmax(H), H.shape)
cx = (xedges[idx[0]] + xedges[idx[0]+1]) / 2
cy = (yedges[idx[1]] + yedges[idx[1]+1]) / 2

print(f"Эмпирический центр Leo A: X = {cx:.1f}, Y = {cy:.1f}")

# 2. Расстояние от центра для каждой звезды
R = np.sqrt((x_la - cx)**2 + (y_la - cy)**2)

# 3. Радиальные бины (до R=1800 пикселей, дальше края матрицы сильно искажают площадь кольца)
bins = np.linspace(0, 1800, 20)
bin_centers = (bins[:-1] + bins[1:]) / 2
areas = np.pi * (bins[1:]**2 - bins[:-1]**2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

unique_labels = np.unique(label_leoa)
colors = plt.cm.tab10.colors

for i, lbl in enumerate(unique_labels):
    mask = (label_leoa == lbl)
    stage_name = parsec_abbr.get(int(lbl), f'Stage {lbl}')
    
    # Кол-во звезд в кольце
    counts, _ = np.histogram(R[mask], bins=bins)
    
    # Поверхностная плотность (звезд на 10^6 кв. пикселей)
    density = (counts / areas) * 1e6
    
    # График 1: Абсолютная плотность
    axes[0].plot(bin_centers, density, label=stage_name, color=colors[i%10], lw=2, marker='o', markersize=4)
    
    # График 2: Нормированная плотность (чтобы сравнить профили)
    # Нормируем на центральную плотность (первый бин)
    if density[0] > 0:
        axes[1].plot(bin_centers, density / density[0], label=stage_name, color=colors[i%10], lw=2, marker='o', markersize=4)

axes[0].set_title("Абсолютная плотность звезд", fontsize=14)
axes[0].set_ylabel("Плотность (звезд / 10$^6$ пикс$^2$)", fontsize=12)

axes[1].set_title("Нормированный профиль (к центру = 1)", fontsize=14)
axes[1].set_ylabel("Относительная концентрация", fontsize=12)

for ax in axes:
    ax.set_xlabel("Расстояние от центра R (пиксели)", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(title="Стадия")
    ax.set_xlim(0, 1800)

plt.tight_layout()
plt.show()



In [ ]:
# Старый профиль IC 1613 по локальным x, y и по кольцам больше не используется:
# пиксели разных полей не в одной системе, а кольца режут края кадров.
# Центры стадий и спад плотности — в пункте 11, по df_ic1613_in.
print('Профиль IC 1613 считается в пункте 11 по звёздам внутри параллелограммов.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.spatial import cKDTree

# ══════════════════════════════════════════════════════════
# ПАРАМЕТРЫ
# ══════════════════════════════════════════════════════════
MH_LEOA  = -1.4   # металличность Leo A в iso_sum.dat

# Порог расстояния в CMD-пространстве.
# Метрика: sqrt(ΔCol² + (0.4·ΔMag)²)  ← коэффициент 0.4 нужен, чтобы
# диапазон цвета (~4 маг) и блеска (~10 маг) вносили сопоставимый вклад.
# MAX_DIST = 0.5 ≈ ширина RGB-последовательности плюс запас на фотошум.
# Звёзды дальше этого порога — не классифицируются (выводятся серыми).
MAX_DIST = 0.5
C_SCALE, M_SCALE = 1.0, 0.4

# ══════════════════════════════════════════════════════════
# 1. KD-дерево по сетке изохрон Leo A
# ══════════════════════════════════════════════════════════
iso_la   = iso_sum[np.isclose(iso_sum['MH'], MH_LEOA)].copy()
iso_ages = iso_la['logAge'].to_numpy()

grid_pts = np.column_stack([
    iso_la['color_mag'].to_numpy() * C_SCALE,
    iso_la['F814Wmag'].to_numpy()  * M_SCALE
])
tree = cKDTree(grid_pts)

# ══════════════════════════════════════════════════════════
# 2. Ближайшая изохрона для каждой звезды
# ══════════════════════════════════════════════════════════
c_obs = color_abs_leoa.to_numpy()
m_obs = mag_F814W_abs_leoa.to_numpy()
obs_pts = np.column_stack([c_obs * C_SCALE, m_obs * M_SCALE])

dist_arr, idx_arr = tree.query(obs_pts, workers=-1)
age_arr  = iso_ages[idx_arr]
too_far  = dist_arr > MAX_DIST

n_total = len(dist_arr)
n_far   = too_far.sum()
print(f"Всего звёзд Leo A:                      {n_total:,}")
print(f"Не классифицировано (d > {MAX_DIST}):      {n_far:,}  ({100*n_far/n_total:.1f}%)")
print(f"Классифицировано по ближайшей изохроне: {n_total-n_far:,}  ({100*(1-n_far/n_total):.1f}%)")
print()

unique_ages = sorted(iso_la['logAge'].unique())
print("Распределение по возрастам:")
for ua in unique_ages:
    n = ((age_arr == ua) & ~too_far).sum()
    yr = 10**ua
    lbl = f"{yr/1e9:.1f} Gyr" if yr >= 1e9 else f"{yr/1e6:.0f} Myr"
    print(f"  logAge {ua:.1f}  ({lbl:>8s}):  {n:,}")

# ══════════════════════════════════════════════════════════
# Цвета: plasma от молодых (жёлтый) к старым (фиолетовый)
# ══════════════════════════════════════════════════════════
n_ages = len(unique_ages)
cmap_base  = plt.colormaps['plasma'].resampled(n_ages)
age_colors = {a: cmap_base(i / (n_ages - 1)) for i, a in enumerate(unique_ages)}

def age_label(la):
    yr = 10**la
    return f"{yr/1e9:.1f} Gyr" if yr >= 1e9 else f"{yr/1e6:.0f} Myr"

# ══════════════════════════════════════════════════════════
# 3. CMD по возрасту
# ══════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 9))
ax.scatter(c_obs[too_far], m_obs[too_far], s=0.3, c='lightgray', alpha=0.25, zorder=1, label='нет данных')
for ua in reversed(unique_ages):          # молодые сверху
    m = (age_arr == ua) & ~too_far
    if m.sum() == 0: continue
    ax.scatter(c_obs[m], m_obs[m], s=0.8, color=age_colors[ua], alpha=0.6, zorder=2, label=age_label(ua))
ax.invert_yaxis()
ax.set_xlim(-1, 4);  ax.set_ylim(2, -8)
ax.set_xlabel('F475W − F814W (абс.)', fontsize=12)
ax.set_ylabel('F814W (абс.)', fontsize=12)
ax.set_title('Leo A — CMD по возрасту ближайшей изохроны', fontsize=13)
ax.grid(True, alpha=0.25)
ax.legend(title='Возраст', fontsize=7, markerscale=5, loc='upper right', ncol=2, framealpha=0.85)
plt.tight_layout();  plt.show()

# ══════════════════════════════════════════════════════════
# 4. Пространственное распределение по возрасту
# ══════════════════════════════════════════════════════════
x_la = df_leoa['x'].to_numpy()
y_la = df_leoa['y'].to_numpy()

H, xe, ye = np.histogram2d(x_la, y_la, bins=100)
ix, iy = np.unravel_index(np.argmax(H), H.shape)
cx = 0.5*(xe[ix]+xe[ix+1]);  cy = 0.5*(ye[iy]+ye[iy+1])

fig2, ax2 = plt.subplots(figsize=(8, 8))
ax2.scatter(x_la[too_far], y_la[too_far], s=0.3, c='lightgray', alpha=0.2, zorder=1)
for ua in reversed(unique_ages):
    m = (age_arr == ua) & ~too_far
    if m.sum() == 0: continue
    ax2.scatter(x_la[m], y_la[m], s=0.8, color=age_colors[ua], alpha=0.5, zorder=2, label=age_label(ua))
ax2.plot(cx, cy, 'k+', ms=18, mew=2.5, zorder=5, label='Центр')
ax2.set_xlabel('X (пиксели)', fontsize=12)
ax2.set_ylabel('Y (пиксели)', fontsize=12)
ax2.set_title('Leo A — пространственное распределение по возрасту', fontsize=13)
ax2.set_aspect('equal')
ax2.legend(title='Возраст', fontsize=7, markerscale=5, ncol=2, framealpha=0.85)
ax2.grid(True, alpha=0.25)
plt.tight_layout();  plt.show()

# ══════════════════════════════════════════════════════════
# 5. Радиальное распределение по возрасту (subplots по пикселям)
# ══════════════════════════════════════════════════════════
R = np.sqrt((x_la - cx)**2 + (y_la - cy)**2)

bins  = np.linspace(0, 1800, 20)
bc    = 0.5*(bins[:-1]+bins[1:])
areas = np.pi*(bins[1:]**2 - bins[:-1]**2)

# Берём только возрасты с достаточным числом звёзд (N ≥ 50)
active_ages = [ua for ua in unique_ages if ((age_arr == ua) & ~too_far).sum() >= 50]

n_cols = 3;  n_rows = math.ceil(len(active_ages) / n_cols)
fig3, axes3 = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes3 = np.atleast_1d(axes3).flatten()

for i, ua in enumerate(active_ages):
    m = (age_arr == ua) & ~too_far
    counts, _ = np.histogram(R[m], bins=bins)
    density   = (counts / areas) * 1e6

    ax = axes3[i]
    ax.plot(bc, density, color=age_colors[ua], lw=2, marker='o', ms=4)
    ax.set_title(f"{age_label(ua)}  (N={m.sum():,})", fontweight='bold', color=age_colors[ua])
    ax.set_xlabel('R (пиксели)');  ax.set_ylabel('зв. / 10⁶ пикс²')
    ax.set_xlim(0, 1800);  ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)

for j in range(i+1, len(axes3)):
    fig3.delaxes(axes3[j])

plt.suptitle('Leo A — радиальное распределение по возрасту (ближайшая изохрона)',
             fontsize=14, y=1.02)
plt.tight_layout();  plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ══════════════════════════════════════════════════════════
# Наклон радиального профиля: фит ln(Σ) = a - R/h
# ══════════════════════════════════════════════════════════
bins  = np.linspace(0, 1800, 20)
bc    = 0.5*(bins[:-1]+bins[1:])
areas = np.pi*(bins[1:]**2 - bins[:-1]**2)

# Фитируем только бины внутри [R_min, R_max], где плотность > 0
R_FIT_MIN = 100   # пикс: пропускаем самый центр (нерешённое ядро, края кадра)
R_FIT_MAX = 1400  # пикс: дальше начинаются края поля → плотность падает артефактно

results = []   # [(logAge, N, h, h_err, slope, slope_err)]

for ua in unique_ages:
    m = (age_arr == ua) & ~too_far
    N = m.sum()
    if N < 50:
        results.append((ua, N, np.nan, np.nan, np.nan, np.nan))
        continue

    counts, _ = np.histogram(R[m], bins=bins)
    density   = (counts / areas) * 1e6

    # Выбираем бины для фита
    mask_fit = (bc >= R_FIT_MIN) & (bc <= R_FIT_MAX) & (density > 0)
    if mask_fit.sum() < 3:
        results.append((ua, N, np.nan, np.nan, np.nan, np.nan))
        continue

    x_fit = bc[mask_fit]
    y_fit = np.log(density[mask_fit])

    res = linregress(x_fit, y_fit)
    slope     = res.slope          # = -1/h
    slope_err = res.stderr
    h         = -1.0 / slope if slope < 0 else np.nan
    h_err     = (slope_err / slope**2) if slope < 0 else np.nan

    results.append((ua, N, h, h_err, slope, slope_err))

# ──────────────────────────────────────────────────────────
# Вывод таблицы в ячейку
# ──────────────────────────────────────────────────────────
print(f"{'logAge':>8}  {'Возраст':>10}  {'N':>7}  {'h (пикс)':>10}  {'±':>6}  {'slope·10⁻³':>11}")
print("─"*65)
for ua, N, h, h_err, slope, slope_err in results:
    lbl = age_label(ua)
    if np.isnan(h):
        print(f"  {ua:.1f}   {lbl:>10}   {N:>7}   {'—':>10}")
    else:
        print(f"  {ua:.1f}   {lbl:>10}   {N:>7}   {h:>10.1f}  {h_err:>6.1f}  {slope*1e3:>+10.2f}")

# ══════════════════════════════════════════════════════════
# График 1: масштабная длина h vs logAge
# ══════════════════════════════════════════════════════════
ages_ok = np.array([r[0] for r in results if not np.isnan(r[2])])
h_ok    = np.array([r[2] for r in results if not np.isnan(r[2])])
h_err_ok= np.array([r[3] for r in results if not np.isnan(r[2])])
N_ok    = np.array([r[1] for r in results if not np.isnan(r[2])])
colors_ok = [age_colors[a] for a in ages_ok]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
sc = ax.scatter(ages_ok, h_ok, c=[age_colors[a] for a in ages_ok],
                s=80+N_ok/300, zorder=3, edgecolors='k', linewidths=0.5)
ax.errorbar(ages_ok, h_ok, yerr=h_err_ok, fmt='none', ecolor='gray',
            elinewidth=1.2, capsize=4, zorder=2)


ax.set_xlabel('logAge', fontsize=12)
ax.set_ylabel('Масштабная длина  h  (пиксели)', fontsize=12)
ax.set_title('Leo A — h(R) vs возраст\n(меньше h → компактнее)', fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.0, 5_000.0)

# Вторая ось X: реальные возрасты
ax2x = ax.twiny()
ax2x.set_xlim(ax.get_xlim())
tick_la  = [7.0, 7.6, 8.0, 8.6, 9.0, 9.6, 10.0]
tick_lbl = ['10M','40M','100M','400M','1G','4G','10G']
ax2x.set_xticks(tick_la)
ax2x.set_xticklabels(tick_lbl, fontsize=8)
ax2x.set_xlabel('Возраст', fontsize=10)

# ══════════════════════════════════════════════════════════
# График 2: все профили на одном — нормированные по h
# ══════════════════════════════════════════════════════════
ax2 = axes[1]

for ua, N, h, h_err, slope, slope_err in results:
    if np.isnan(h) or N < 50:
        continue
    m = (age_arr == ua) & ~too_far
    counts, _ = np.histogram(R[m], bins=bins)
    density   = (counts / areas) * 1e6
    mask_nz   = density > 0
    # Нормируем R на h, плотность — на центральную
    R_norm = bc[mask_nz] / h
    D_norm = density[mask_nz] / density[mask_nz][0]
    ax2.plot(R_norm, D_norm, color=age_colors[ua], lw=1.5, alpha=0.8, label=age_label(ua))

ax2.set_xlabel('R / h', fontsize=12)
ax2.set_ylabel('Σ / Σ(R_min)', fontsize=12)
ax2.set_title('Leo A — нормированные профили\n(все возрасты на одном графике)', fontsize=12)
ax2.set_xlim(0, 5); ax2.set_ylim(bottom=0)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=7, ncol=2, title='Возраст')

plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ══════════════════════════════════════════════════════════
# Интерактивная карта Leo A: галочки по возрасту
# ══════════════════════════════════════════════════════════
palette = [age_colors[ua] for ua in unique_ages]

def rgba(c, a=0.6):
    r,g,b,_ = c
    return f'rgba({int(r*255)},{int(g*255)},{int(b*255)},{a})'

traces = []
trace_map = {}   # logAge → индекс трейса

# Неклассифицированные
traces.append(go.Scattergl(
    x=x_la[too_far], y=y_la[too_far], mode='markers',
    marker=dict(size=1.5, color='lightgray', opacity=0.3),
    name='нет данных', visible=True
))

for i, ua in enumerate(unique_ages):
    m = (age_arr == ua) & ~too_far
    if m.sum() == 0:
        continue
    traces.append(go.Scattergl(
        x=x_la[m], y=y_la[m], mode='markers',
        marker=dict(size=2, color=rgba(age_colors[ua], 0.6), opacity=0.7),
        name=age_label(ua),
        visible=True
    ))
    trace_map[ua] = len(traces) - 1

# Центр
traces.append(go.Scattergl(
    x=[cx], y=[cy], mode='markers',
    marker=dict(size=16, color='black', symbol='cross',
                line=dict(width=2, color='white')),
    name='Центр', visible=True
))
center_idx = len(traces) - 1

fig_w = go.FigureWidget(traces)
fig_w.update_layout(
    xaxis_title='X (пиксели)', yaxis_title='Y (пиксели)',
    title='Leo A — пространственное распределение по возрасту',
    width=700, height=650, dragmode='pan',
    legend=dict(itemsizing='constant')
)

# ─── Галочки ───────────────────────────────────────────
checkboxes = []

# «Нет данных»
cb_nd = widgets.Checkbox(value=True, description='нет данных', indent=False,
                          layout=widgets.Layout(width='110px'))
def toggle_nd(change):
    fig_w.data[0].visible = change['new']
cb_nd.observe(toggle_nd, names='value')
checkboxes.append(cb_nd)

# «Все» / «Никого»
btn_all  = widgets.Button(description='✓ Все',  button_style='success',
                           layout=widgets.Layout(width='80px', height='28px'))
btn_none = widgets.Button(description='✗ Нет',  button_style='danger',
                           layout=widgets.Layout(width='80px', height='28px'))

age_checkboxes = []
for ua in unique_ages:
    if ua not in trace_map:
        continue
    cb = widgets.Checkbox(
        value=True,
        description=age_label(ua),
        indent=False,
        style={'description_width': '0px'},
        layout=widgets.Layout(width='110px')
    )
    # цветная метка через HTML
    r,g,b,_ = age_colors[ua]
    color_hex = f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'
    cb = widgets.HBox([
        widgets.HTML(f'<span style="color:{color_hex};font-size:18px">■</span>'),
        widgets.Checkbox(value=True, description=age_label(ua), indent=False,
                         layout=widgets.Layout(width='100px'))
    ])
    real_cb = cb.children[1]

    def make_toggle(lbl):
        def toggle(change):
            fig_w.data[trace_map[lbl]].visible = change['new']
        return toggle

    real_cb.observe(make_toggle(ua), names='value')
    age_checkboxes.append((ua, real_cb, cb))
    checkboxes.append(cb)

def on_all(b):
    for ua, real_cb, _ in age_checkboxes:
        real_cb.value = True
def on_none(b):
    for ua, real_cb, _ in age_checkboxes:
        real_cb.value = False

btn_all.on_click(on_all)
btn_none.on_click(on_none)

# ─── Сборка layout ─────────────────────────────────────
header  = widgets.HBox([btn_all, btn_none])
cb_list = widgets.VBox([header] + [cb for cb in checkboxes],
                        layout=widgets.Layout(overflow_y='auto', max_height='620px'))

display(widgets.HBox([fig_w, cb_list]))

---
## 11. IC 1613 — центры популяций и спад плотности

Центры стадий — пик сглаженной двумерной плотности в (RA, Dec), по каталогу без повторов полей.
Выборка та же, что на карте пункта 8: `df_ic1613_in`, только звёзды внутри параллелограмма своего поля.

Спад плотности считается не кольцами. У каждой звезды локальная плотность — по расстоянию до k-го соседа той же стадии. Кривая ρ(R) — ядерное сглаживание этих плотностей по расстоянию до центра своей стадии. Звёзды у края мозаики, чьё соседство обрезано границей, в кривую не входят.


In [ ]:
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree, ConvexHull

if 'df_ic1613_in' not in dir() or len(df_ic1613_in) != len(label_ic1613_in):
    raise RuntimeError('нет df_ic1613_in — перезапусти ячейку пункта 8 с параллелограммами')
df_ic_in = df_ic1613_in
lab_in = np.asarray(label_ic1613_in)
print(f'выборка пункта 8, внутри параллелограммов: {len(df_ic_in):,}')

_dec0 = float(df_ic_in['dec'].mean())
_ra0 = float(df_ic_in['ra'].mean())
_cosd = np.cos(np.deg2rad(_dec0))
# плоские координаты, угловые минуты; ось x растёт к востоку
x_all = (df_ic_in['ra'].to_numpy() - _ra0) * _cosd * 60.0
y_all = (df_ic_in['dec'].to_numpy() - _dec0) * 60.0
lab_all = lab_in

# расстояние внутрь от границы покрытия (выпуклая оболочка всех звёзд)
_hull = ConvexHull(np.column_stack([x_all, y_all]))
_hv = np.column_stack([x_all, y_all])[_hull.vertices]

def _clearance(x, y, hv=_hv):
    """Минимальное расстояние до стороны оболочки, угл. мин."""
    a = hv
    b = np.roll(hv, -1, axis=0)
    ab = b - a
    ab2 = np.sum(ab * ab, axis=1)
    best = np.full(len(x), np.inf)
    pts = np.column_stack([x, y])
    for i in range(len(a)):
        t = np.clip(((pts - a[i]) @ ab[i]) / ab2[i], 0.0, 1.0)
        proj = a[i] + t[:, None] * ab[i]
        best = np.minimum(best, np.hypot(pts[:, 0] - proj[:, 0], pts[:, 1] - proj[:, 1]))
    return best

clear_all = _clearance(x_all, y_all)

stage_ids = [int(v) for v in np.unique(lab_all) if int(v) >= 0]
stage_colors = plt.cm.tab10.colors

def _density_peak(x, y, bins=90, sigma=1.6):
    H, xe, ye = np.histogram2d(x, y, bins=bins)
    Hs = gaussian_filter(H.astype(float), sigma=sigma)
    ix, iy = np.unravel_index(np.argmax(Hs), Hs.shape)
    return 0.5 * (xe[ix] + xe[ix + 1]), 0.5 * (ye[iy] + ye[iy + 1])

centers = {}
for i, lbl in enumerate(stage_ids):
    m = lab_all == lbl
    if m.sum() < 80:
        continue
    cx, cy = _density_peak(x_all[m], y_all[m])
    centers[lbl] = (cx, cy, int(m.sum()), stage_colors[i % 10])
    ra_c = _ra0 + cx / (_cosd * 60.0)
    dec_c = _dec0 + cy / 60.0
    name = parsec_abbr.get(lbl, f'Stage {lbl}')
    print(f'{name:6s}  N={m.sum():7,}   центр  α={ra_c:.5f}°  δ={dec_c:+.5f}°')

# --- карта ---
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(x_all, y_all, s=0.15, c='0.75', alpha=0.35, linewidths=0,
           rasterized=True, zorder=1, label='все звёзды, без повторов')
for lbl, (cx, cy, n, col) in centers.items():
    name = parsec_abbr.get(lbl, f'Stage {lbl}')
    ax.scatter([cx], [cy], s=220, c=[col], edgecolors='black', linewidths=1.4,
               zorder=5, label=f'{name}  (N={n:,})')
ax.set_aspect('equal', adjustable='datalim')
ax.invert_xaxis()
ax.set_xlabel('Δα cos δ (угл. мин)')
ax.set_ylabel('Δδ (угл. мин)')
ax.set_title('IC 1613: центры стадий — пик плотности')
ax.legend(markerscale=1.0, fontsize=8, loc='best', framealpha=0.95)
plt.tight_layout()
plt.show()

# --- спад плотности без колец ---
# ρ = k / (π d_k²), d_k — расстояние до k-го соседа той же стадии.
# ρ(R) — гауссово среднее этих ρ по расстоянию до центра стадии.
K_NN = 25
BAND_ARCMIN = 0.35

fig, ax = plt.subplots(figsize=(9, 6))
for lbl, (cx, cy, n, col) in centers.items():
    m = lab_all == lbl
    if m.sum() < K_NN + 5:
        continue
    xx, yy = x_all[m], y_all[m]
    tree = cKDTree(np.column_stack([xx, yy]))
    dk = tree.query(np.column_stack([xx, yy]), k=K_NN + 1)[0][:, -1]
    rho = K_NN / (np.pi * np.maximum(dk, 1e-4) ** 2)   # звёзд / угл. мин²
    rad = np.hypot(xx - cx, yy - cy)
    ok = clear_all[m] > dk
    if ok.sum() < 50:
        continue
    rad, rho = rad[ok], rho[ok]
    r_max = np.percentile(rad, 95)
    r_grid = np.linspace(0.0, r_max, 70)
    scale = BAND_ARCMIN
    # ядерная регрессия, не гистограмма по кольцам
    log_rho = np.empty_like(r_grid)
    for j, rg in enumerate(r_grid):
        w = np.exp(-0.5 * ((rad - rg) / scale) ** 2)
        sw = w.sum()
        log_rho[j] = np.log10(np.sum(w * rho) / sw) if sw > 0 else np.nan
    name = parsec_abbr.get(lbl, f'Stage {lbl}')
    ax.plot(r_grid, log_rho, color=col, lw=2.2, label=name)

ax.set_xlabel('R от центра своей стадии (угл. мин)')
ax.set_ylabel('log10  ρ   (звёзд / угл. мин²)')
ax.set_title(f'IC 1613: спад плотности (k={K_NN} сосед, окно {BAND_ARCMIN}′)')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
# Тут будем изучать распределение звезд вдоль оси

# Большая ось выборки df_ic1613_in в тех же плоских координатах, что и карта пункта 11.
# Отрезок — от края до края покрытия вдоль этой оси; он проходит через все 6 полей.
if 'df_ic1613_in' not in dir():
    raise RuntimeError('нет df_ic1613_in — перезапусти ячейку пункта 8')

_dec0 = float(df_ic1613_in['dec'].mean())
_ra0 = float(df_ic1613_in['ra'].mean())
_cosd = np.cos(np.deg2rad(_dec0))
x = (df_ic1613_in['ra'].to_numpy() - _ra0) * _cosd * 60.0
y = (df_ic1613_in['dec'].to_numpy() - _dec0) * 60.0
xy = np.column_stack([x, y])
center = xy.mean(axis=0)

_, _, _vt = np.linalg.svd(xy - center, full_matrices=False)
axis = _vt[0]   # направление наибольшей дисперсии
proj = (xy - center) @ axis
t_min, t_max = float(proj.min()), float(proj.max())
p0 = center + t_min * axis
p1 = center + t_max * axis

print(f"Большая ось: угол {np.degrees(np.arctan2(axis[1], axis[0])):.1f}°")
print(f"Отрезок: {t_min:.2f} … {t_max:.2f} угл. мин от центра выборки, длина {t_max - t_min:.2f}′")

fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(x, y, s=0.15, c='0.75', alpha=0.35, linewidths=0, rasterized=True, zorder=1)
ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color='black', lw=2.5, zorder=4, label='большая ось')
ax.scatter([center[0]], [center[1]], s=80, c='black', zorder=5, label='центр выборки')
ax.set_aspect('equal', adjustable='datalim')
ax.invert_xaxis()
ax.set_xlabel('Δα cos δ (угл. мин)')
ax.set_ylabel('Δδ (угл. мин)')
ax.set_title('IC 1613: отрезок вдоль большой оси')
ax.legend(loc='best', framealpha=0.95)
plt.tight_layout()
plt.show()


In [ ]:
# Плотность вдоль большой оси. Без дополнительного сглаживания:
# узкая полоса вокруг оси, шаг по оси 0.4 угл. мин.
# Точки и ячейки, которые попадают в зазор между двумя чипами ACS, не используются.

if 'df_ic1613_in' not in dir() or len(df_ic1613_in) != len(label_ic1613_in):
    raise RuntimeError('нет df_ic1613_in — перезапусти ячейку пункта 8')
if 'ic1613_lines' not in dir() or 'IC1613_FIELD_PRIORITY' not in dir():
    raise RuntimeError('нет параллелограммов — перезапусти ячейку пункта 8')

def _chip_gap_y(y, bins=800, half_px=45.0):
    """Зазор ACS: узкая полоса постоянной y вокруг просадки посередине кадра.

    Физический зазор ~50 px. Берём ±45 px вокруг минимума сглаженной
    гистограммы, чтобы не захватывать наполовину заполненные края.
    """
    hist, edges = np.histogram(y, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    kernel = np.array([1.0, 2.0, 3.0, 2.0, 1.0])
    kernel /= kernel.sum()
    smooth = np.convolve(hist.astype(float), kernel, mode='same')
    lo, hi = int(0.30 * bins), int(0.70 * bins)
    i = lo + int(np.argmin(smooth[lo:hi]))
    return float(centers[i] - half_px), float(centers[i] + half_px)

def _field_inverse(sub):
    xx = sub['x'].to_numpy(float)
    yy = sub['y'].to_numpy(float)
    A = np.column_stack([xx, yy, np.ones(len(sub))])
    cra, *_ = np.linalg.lstsq(A, sub['ra'].to_numpy(float), rcond=None)
    cdec, *_ = np.linalg.lstsq(A, sub['dec'].to_numpy(float), rcond=None)
    M = np.array([[cra[0], cra[1]], [cdec[0], cdec[1]]])
    return np.linalg.inv(M), np.array([cra[2], cdec[2]])

_gaps = {}
_inv = {}
for _field, _sub in df_ic1613.groupby('field'):
    _gaps[_field] = _chip_gap_y(_sub['y'].to_numpy())
    _inv[_field] = _field_inverse(_sub)
    print(f'{_field}: зазор чипов y = {_gaps[_field][0]:.0f} … {_gaps[_field][1]:.0f} px')

_in_gap = np.zeros(len(df_ic1613_in), dtype=bool)
for _field, (_ylo, _yhi) in _gaps.items():
    _m = df_ic1613_in['field'].to_numpy() == _field
    _yy = df_ic1613_in['y'].to_numpy()
    _in_gap[_m] = (_yy[_m] >= _ylo) & (_yy[_m] <= _yhi)
print(f'звёзды в зазоре чипов, исключены: {_in_gap.sum():,}')

_dec0 = float(df_ic1613_in['dec'].mean())
_ra0 = float(df_ic1613_in['ra'].mean())
_cosd = np.cos(np.deg2rad(_dec0))
x = (df_ic1613_in['ra'].to_numpy() - _ra0) * _cosd * 60.0
y = (df_ic1613_in['dec'].to_numpy() - _dec0) * 60.0
xy = np.column_stack([x, y])
center = xy.mean(axis=0)
_, _, _vt = np.linalg.svd(xy - center, full_matrices=False)
axis_vec = _vt[0]
normal = np.array([-axis_vec[1], axis_vec[0]])

s = (xy - center) @ axis_vec
d = np.abs((xy - center) @ normal)
lab = np.asarray(label_ic1613_in)

SLIT = 0.75
BIN = 0.40
VLINES = [0.0, 0.5, 1.35]   # общие пунктиры, угл. мин вдоль оси
s_min, s_max = np.percentile(s, [0.5, 99.5])
edges = np.arange(s_min, s_max + BIN, BIN)
centers_s = 0.5 * (edges[:-1] + edges[1:])
area = BIN * (2 * SLIT)

def _gap_fraction(edges, axis_vec, normal, center):
    """Доля ячейки полосы, попавшая в зазор чипов (среди точек внутри кадра)."""
    frac = np.zeros(len(edges) - 1)
    ns, nd = 8, 5
    for j in range(len(frac)):
        ss = np.linspace(edges[j], edges[j + 1], ns, endpoint=False)
        ss += 0.5 * (edges[j + 1] - edges[j]) / ns
        ds = np.linspace(-SLIT, SLIT, nd)
        SS, DD = np.meshgrid(ss, ds, indexing='ij')
        pts = center + SS.ravel()[:, None] * axis_vec + DD.ravel()[:, None] * normal
        ra = _ra0 + pts[:, 0] / (_cosd * 60.0)
        dec = _dec0 + pts[:, 1] / 60.0
        in_gap_pt = np.zeros(len(ra), dtype=bool)
        inside = np.zeros(len(ra), dtype=bool)
        for _field in IC1613_FIELD_PRIORITY:
            _m = in_field_parallelogram(ra, dec, ic1613_lines[_field]) & ~inside
            if np.any(_m):
                Minv, t = _inv[_field]
                y_pix = ((np.column_stack([ra[_m], dec[_m]]) - t) @ Minv.T)[:, 1]
                ylo, yhi = _gaps[_field]
                in_gap_pt[_m] = (y_pix >= ylo) & (y_pix <= yhi)
            inside |= in_field_parallelogram(ra, dec, ic1613_lines[_field])
        frac[j] = in_gap_pt[inside].mean() if np.any(inside) else 0.0
    return frac

gap_fraction = _gap_fraction(edges, axis_vec, normal, center)
keep_bin = gap_fraction < 0.20
print(f'ячейки полосы, задевающие зазор чипов: {(~keep_bin).sum()} из {len(keep_bin)}')

in_slit = (d <= SLIT) & ~_in_gap
stage_ids = [int(v) for v in np.unique(lab) if int(v) >= 0]
colors = plt.cm.tab10.colors
main_stages = {'MS', 'SGB', 'RGB'}

def _draw(ax, wanted, title):
    for i, lbl in enumerate(stage_ids):
        name = parsec_abbr.get(lbl, f'Stage {lbl}')
        if (name in main_stages) != wanted:
            continue
        m = in_slit & (lab == lbl)
        if m.sum() < 30:
            continue
        counts, _ = np.histogram(s[m], bins=edges)
        rho = np.full(len(centers_s), np.nan)
        rho[keep_bin] = counts[keep_bin] / area
        ax.plot(centers_s, rho, color=colors[i % 10], lw=1.6, label=f'{name}  (N={m.sum():,})')
    for _s in VLINES:
        ax.axvline(_s, color='0.15', lw=1.0, ls=':')
    ax.set_ylabel('ρ  (звёзд / угл. мин²)')
    ax.set_title(title)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, ncol=2)

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
_draw(axes[0], True, f'MS, SGB, RGB  —  полоса ±{SLIT}′, шаг {BIN}′, без зазора чипов')
_draw(axes[1], False, 'остальные стадии')
axes[1].set_xlabel('s вдоль большой оси (угл. мин)')
plt.tight_layout()
plt.show()


In [ ]:
# CMD звёзд IC 1613 с 1.0′ ≤ s ≤ 2.0′ вдоль большой оси.
# Та же ось, что в профиле плотности. Берутся все звёзды df_ic1613_in в этом интервале s.

if 'df_ic1613_in' not in dir() or len(df_ic1613_in) == 0:
    raise RuntimeError('нет df_ic1613_in — перезапусти ячейку пункта 8')

S_LO, S_HI = 1.0, 2.0

_dec0 = float(df_ic1613_in['dec'].mean())
_ra0 = float(df_ic1613_in['ra'].mean())
_cosd = np.cos(np.deg2rad(_dec0))
x = (df_ic1613_in['ra'].to_numpy() - _ra0) * _cosd * 60.0
y = (df_ic1613_in['dec'].to_numpy() - _dec0) * 60.0
xy = np.column_stack([x, y])
center = xy.mean(axis=0)
_, _, _vt = np.linalg.svd(xy - center, full_matrices=False)
axis_vec = _vt[0]
s = (xy - center) @ axis_vec

m = (s >= S_LO) & (s <= S_HI)
sub = df_ic1613_in.loc[m]
color = sub['mag_F475W'].to_numpy() - sub['mag_F814W'].to_numpy()
mag = sub['mag_F814W'].to_numpy()
print(f'звёзды с {S_LO:.1f}′ ≤ s ≤ {S_HI:.1f}′: {m.sum():,} из {len(df_ic1613_in):,}')

fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter(color, mag, s=2, alpha=0.35, c='black', linewidths=0)
ax.invert_yaxis()
ax.set_xlabel('mag_F475W − mag_F814W')
ax.set_ylabel('mag_F814W')
ax.set_title(f'CMD IC 1613,  {S_LO:.1f}′ ≤ s ≤ {S_HI:.1f}′   (N={m.sum():,})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Тепловые карты поверхностной плотности по стадиям, в (α, δ).
# Плотность логарифмическая. Зазор между чипами ACS не используется.

if 'df_ic1613_in' not in dir() or len(df_ic1613_in) != len(label_ic1613_in):
    raise RuntimeError('нет df_ic1613_in — перезапусти ячейку пункта 8')
if 'ic1613_lines' not in dir() or 'IC1613_FIELD_PRIORITY' not in dir():
    raise RuntimeError('нет параллелограммов — перезапусти ячейку пункта 8')

def _chip_gap_y(y, bins=800, half_px=45.0):
    hist, edges = np.histogram(y, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    kernel = np.array([1.0, 2.0, 3.0, 2.0, 1.0])
    kernel /= kernel.sum()
    smooth = np.convolve(hist.astype(float), kernel, mode='same')
    lo, hi = int(0.30 * bins), int(0.70 * bins)
    i = lo + int(np.argmin(smooth[lo:hi]))
    return float(centers[i] - half_px), float(centers[i] + half_px)

def _field_inverse(sub):
    xx = sub['x'].to_numpy(float)
    yy = sub['y'].to_numpy(float)
    A = np.column_stack([xx, yy, np.ones(len(sub))])
    cra, *_ = np.linalg.lstsq(A, sub['ra'].to_numpy(float), rcond=None)
    cdec, *_ = np.linalg.lstsq(A, sub['dec'].to_numpy(float), rcond=None)
    M = np.array([[cra[0], cra[1]], [cdec[0], cdec[1]]])
    return np.linalg.inv(M), np.array([cra[2], cdec[2]])

_gaps, _inv = {}, {}
for _field, _sub in df_ic1613.groupby('field'):
    _gaps[_field] = _chip_gap_y(_sub['y'].to_numpy())
    _inv[_field] = _field_inverse(_sub)

_in_gap = np.zeros(len(df_ic1613_in), dtype=bool)
_fields = df_ic1613_in['field'].to_numpy()
_yy = df_ic1613_in['y'].to_numpy()
for _field, (_ylo, _yhi) in _gaps.items():
    _m = _fields == _field
    _in_gap[_m] = (_yy[_m] >= _ylo) & (_yy[_m] <= _yhi)
print(f'пропущено звёзд у зазора чипов: {_in_gap.sum():,}')

ra = df_ic1613_in['ra'].to_numpy()
dec = df_ic1613_in['dec'].to_numpy()
lab = np.asarray(label_ic1613_in)
_dec0 = float(dec.mean())
_cosd = np.cos(np.deg2rad(_dec0))

BIN_AMIN = 0.22
bin_dec = BIN_AMIN / 60.0
bin_ra = bin_dec / _cosd          # одинаковый шаг на небе, угл. мин
pad_ra, pad_dec = bin_ra, bin_dec
xe = np.arange(np.floor((ra.min() - pad_ra) / bin_ra) * bin_ra,
               np.ceil((ra.max() + pad_ra) / bin_ra) * bin_ra + 0.5 * bin_ra, bin_ra)
ye = np.arange(np.floor((dec.min() - pad_dec) / bin_dec) * bin_dec,
               np.ceil((dec.max() + pad_dec) / bin_dec) * bin_dec + 0.5 * bin_dec, bin_dec)
area = (xe[1] - xe[0]) * _cosd * 60.0 * (ye[1] - ye[0]) * 60.0   # угл. мин²

xc = 0.5 * (xe[:-1] + xe[1:])
yc = 0.5 * (ye[:-1] + ye[1:])
XX, YY = np.meshgrid(xc, yc, indexing='ij')
ra_g = XX.ravel()
dec_g = YY.ravel()
gap_bin = np.zeros(ra_g.shape, dtype=bool)
taken = np.zeros(ra_g.shape, dtype=bool)
for _field in IC1613_FIELD_PRIORITY:
    _m = in_field_parallelogram(ra_g, dec_g, ic1613_lines[_field]) & ~taken
    if np.any(_m):
        Minv, t = _inv[_field]
        y_pix = ((np.column_stack([ra_g[_m], dec_g[_m]]) - t) @ Minv.T)[:, 1]
        ylo, yhi = _gaps[_field]
        gap_bin[_m] = (y_pix >= ylo) & (y_pix <= yhi)
    taken |= in_field_parallelogram(ra_g, dec_g, ic1613_lines[_field])
gap_bin = gap_bin.reshape(XX.shape)

stage_ids = [int(v) for v in np.unique(lab) if int(v) >= 0 and ((lab == v) & ~_in_gap).sum() >= 20]
ncol = 2
nrow = int(np.ceil(len(stage_ids) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.6 * ncol, 5.0 * nrow), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()
cmap = plt.cm.inferno.copy()
cmap.set_bad('#d9d9d9')
cmap.set_under('#1a1a1a')

for ax, lbl in zip(axes, stage_ids):
    m = (lab == lbl) & ~_in_gap
    H, _, _ = np.histogram2d(ra[m], dec[m], bins=[xe, ye])
    rho = H / area
    log_rho = np.full(rho.shape, np.nan)
    pos = rho > 0
    log_rho[pos] = np.log10(rho[pos])
    finite = log_rho[pos & ~gap_bin]
    vmin = np.percentile(finite, 2) if finite.size else 0.0
    vmax = np.percentile(finite, 98) if finite.size else 1.0
    if vmax <= vmin:
        vmax = vmin + 1.0
    log_rho[(rho == 0) & ~gap_bin] = vmin - 1.0
    log_rho[gap_bin] = np.nan
    im = ax.imshow(
        log_rho.T, origin='lower', extent=[xe[0], xe[-1], ye[0], ye[-1]],
        cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest', aspect='equal',
    )
    name = parsec_abbr.get(lbl, f'Stage {lbl}')
    ax.set_title(f'{name}   N={m.sum():,}', fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='log₁₀ ρ  (звёзд / угл. мин²)')

for ax in axes[len(stage_ids):]:
    ax.axis('off')

for ax in axes:
    if ax.axison:
        ax.set_xlabel('α = RA (deg)')
        ax.set_ylabel('δ = Dec (deg)')
axes[0].invert_xaxis()
fig.suptitle('IC 1613: log плотности в (α, δ). Серым — зазор чипов ACS', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Распределение DOLPHOT object type (колонка type).
# 1 яркая звезда, 2 слабая, 3 вытянутая, 4 горячий пиксель, 5 протяжённый объект.

TYPE_LABELS = {
    1: '1 яркая звезда',
    2: '2 слабая',
    3: '3 вытянутая',
    4: '4 горячий пиксель',
    5: '5 протяжённый',
}

def _type_counts(series):
    s = series.dropna().astype(int)
    counts = s.value_counts()
    rows = []
    for code in range(1, 6):
        n = int(counts.get(code, 0))
        rows.append((code, n))
    other = int(s[~s.isin(range(1, 6))].shape[0]) if len(s) else 0
    return rows, other, int(len(s))

print(f'{"тип":22s} {"IC 1613":>14s} {"Leo A":>14s}')
ic_rows, ic_other, ic_n = _type_counts(df_ic1613['type'])
la_rows, la_other, la_n = _type_counts(df_leoa['type'])
for (code, n_ic), (_, n_la) in zip(ic_rows, la_rows):
    name = TYPE_LABELS[code]
    print(f'{name:22s} {n_ic:8,} ({100 * n_ic / ic_n:5.2f}%) {n_la:8,} ({100 * n_la / la_n:5.2f}%)')
if ic_other or la_other:
    print(f'{"другое":22s} {ic_other:8,} {la_other:8,}')
print(f'{"всего":22s} {ic_n:8,} {la_n:8,}')
print(f'горячих пикселей (type=4):  IC 1613 {dict(ic_rows)[4]:,}   Leo A {dict(la_rows)[4]:,}')

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(1, 6)
w = 0.38
ax.bar(x - w / 2, [n for _, n in ic_rows], width=w, label=f'IC 1613  (N={ic_n:,})', color='C0')
ax.bar(x + w / 2, [n for _, n in la_rows], width=w, label=f'Leo A  (N={la_n:,})', color='C1')
ax.set_xticks(x)
ax.set_xticklabels([TYPE_LABELS[i] for i in x], rotation=15, ha='right')
ax.set_ylabel('число объектов')
ax.set_title('DOLPHOT object type')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Leo A: те же радиальные профили, только MS и RGB.

x_la = df_leoa['x'].to_numpy()
y_la = df_leoa['y'].to_numpy()

H, xe, ye = np.histogram2d(x_la, y_la, bins=100)
ix, iy = np.unravel_index(np.argmax(H), H.shape)
cx = 0.5 * (xe[ix] + xe[ix + 1])
cy = 0.5 * (ye[iy] + ye[iy + 1])
print(f"Центр Leo A: X = {cx:.1f}, Y = {cy:.1f}")

R = np.sqrt((x_la - cx)**2 + (y_la - cy)**2)

bins = np.linspace(0, 1800, 20)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
areas = np.pi * (bins[1:]**2 - bins[:-1]**2)

unique_labels = np.unique(label_leoa)
colors = plt.cm.tab10.colors
color_of = {int(lbl): colors[i % 10] for i, lbl in enumerate(unique_labels)}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, lbl in zip(axes, (1, 3)):
    mask = label_leoa == lbl
    stage = parsec_abbr.get(lbl, f'Stage {lbl}')
    counts, _ = np.histogram(R[mask], bins=bins)
    density = (counts / areas) * 1e6

    ax.plot(bin_centers, density, color=color_of.get(lbl, 'C0'), lw=2, marker='o', ms=4)
    ax.set_title(f"{stage}  (N = {int(mask.sum())})", fontweight='bold')
    ax.set_xlabel("R (пиксели)")
    ax.set_ylabel("плотность (зв. / 10⁶ пикс²)")
    ax.set_xlim(0, 1800)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)

plt.suptitle("Leo A — радиальное распределение: MS и RGB", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Наклоны прямых для профилей MS и RGB из ячейки выше.
# density = a * R + b, R в пикселях, плотность в звёздах / 10^6 пикс².

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, lbl in zip(axes, (1, 3)):
    mask = label_leoa == lbl
    stage = parsec_abbr.get(lbl, f'Stage {lbl}')
    counts, _ = np.histogram(R[mask], bins=bins)
    density = (counts / areas) * 1e6
    a, b = np.polyfit(bin_centers, density, 1)

    ax.plot(bin_centers, density, color=color_of.get(lbl, 'C0'), lw=2, marker='o', ms=4)
    ax.plot(bin_centers, a * bin_centers + b, color='k', lw=1.5, ls='--',
            label=f'a = {a:.4g}')
    ax.set_title(f"{stage}  (N = {int(mask.sum())})", fontweight='bold')
    ax.set_xlabel("R (пиксели)")
    ax.set_ylabel("плотность (зв. / 10⁶ пикс²)")
    ax.set_xlim(0, 1800)
    ax.set_ylim(bottom=0)
    ax.legend()
    ax.grid(True, alpha=0.3)
    print(f"{stage:4s}  наклон a = {a:.6g}  (зв. / 10^6 пикс²) / пикс")

plt.suptitle("Leo A — прямые к профилям MS и RGB", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Те же профили MS и RGB, радиус в килопарсеках.
# Поле ACS/WFC: 202″ на сторону при формате 4096 пикселей
# (STScI ACS Instrument Handbook).
# Линейный масштаб: θ в радианах × расстояние до Leo A.
# Плотность нормирована на первое кольцо.
# Погрешность точки: пуассон √N/S и разброс плотности по четырём секторам
# кольца сверх пуассона (асимметрия и отклонение от круговых колец).

ACS_FOV_ARCSEC = 202.0
ACS_FOV_PIX = 4096.0
arcsec_per_pix = ACS_FOV_ARCSEC / ACS_FOV_PIX
pc_per_pix = dist_leoa * 1000.0 * arcsec_per_pix / 206265.0
kpc_per_pix = pc_per_pix / 1000.0
print(f"Leo A: {dist_leoa:.0f} кпк")
print(f"масштаб: {arcsec_per_pix:.5f}″/пикс,  {pc_per_pix:.4f} пк/пикс")
print(f"R = 1800 пикс → {1800 * kpc_per_pix:.3f} кпк")

R_kpc = bin_centers * kpc_per_pix
areas_pc = areas * pc_per_pix ** 2

N_SEC = 4
phi = np.arctan2(y_la - cy, x_la - cx)
phi_edges = np.linspace(-np.pi, np.pi, N_SEC + 1)
sector_area = areas_pc / N_SEC

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, lbl in zip(axes, (1, 3)):
    mask = label_leoa == lbl
    stage = parsec_abbr.get(lbl, f'Stage {lbl}')
    counts, _ = np.histogram(R[mask], bins=bins)
    density = counts / areas_pc
    sigma_poi = np.sqrt(np.maximum(counts, 1.0)) / areas_pc

    sec_counts = []
    for k in range(N_SEC):
        right = phi < phi_edges[k + 1] if k < N_SEC - 1 else phi <= phi_edges[k + 1]
        in_sec = mask & (phi >= phi_edges[k]) & right
        c_sec, _ = np.histogram(R[in_sec], bins=bins)
        sec_counts.append(c_sec)
    sec_counts = np.vstack(sec_counts)
    sec_rho = sec_counts / sector_area
    var_obs = sec_rho.var(axis=0, ddof=1)
    var_poi = np.mean(sec_counts / sector_area ** 2, axis=0)
    sigma_asym = np.sqrt(np.maximum(var_obs - var_poi, 0.0) / N_SEC)
    sigma = np.sqrt(sigma_poi ** 2 + sigma_asym ** 2)

    rho0 = density[0]
    sig0 = sigma[0]
    norm = density / rho0
    sigma_n = np.sqrt((sigma / rho0) ** 2 + (density * sig0 / rho0 ** 2) ** 2)
    sigma_n[0] = sig0 / rho0
    yerr = np.vstack([np.minimum(sigma_n, norm), sigma_n])
    a, b = np.polyfit(R_kpc, norm, 1)

    ax.errorbar(R_kpc, norm, yerr=yerr, color=color_of.get(lbl, 'C0'),
                lw=1.2, marker='o', ms=4, capsize=2, elinewidth=1)
    ax.plot(R_kpc, a * R_kpc + b, color='k', lw=1.5, ls='--',
            label=f'a = {a:.4g}')
    ax.set_title(f"{stage}  (N = {int(mask.sum())})", fontweight='bold')
    ax.set_xlabel("R (кпк)")
    ax.set_ylabel("плотность / первая точка")
    ax.set_xlim(0, 1800 * kpc_per_pix)
    ax.set_ylim(bottom=0)
    ax.legend()
    ax.grid(True, alpha=0.3)
    print(f"{stage:4s}  ρ₀ = {rho0:.6g} ± {sig0:.3g} зв./пк²   "
          f"наклон a = {a:.6g}  1/кпк")

plt.suptitle("Leo A — прямые к профилям MS и RGB", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Leo A: профили MS и RGB по точкам, а не по кольцам.
# На каждом радиусе r — N_PROBE точек, равномерно по кругу вокруг центра.
# В каждой точке локальная плотность: звёзды стадии в круге радиуса H_AP / площадь.
# Значение — среднее по точкам, погрешность — ошибка среднего std / √n.
# Точки, чей круг выходит за край поля или задевает зазор между чипами ACS, отброшены.
# Нормировка на первую точку, её погрешность перенесена в остальные.

from scipy.spatial import cKDTree, ConvexHull

N_PROBE = 10
H_AP = 60.0          # радиус апертуры, пикс
r_grid = 0.5 * (bins[:-1] + bins[1:])

x_la = df_leoa['x'].to_numpy()
y_la = df_leoa['y'].to_numpy()
H2, xe2, ye2 = np.histogram2d(x_la, y_la, bins=100)
ix2, iy2 = np.unravel_index(np.argmax(H2), H2.shape)
cx0 = 0.5 * (xe2[ix2] + xe2[ix2 + 1])
cy0 = 0.5 * (ye2[iy2] + ye2[iy2 + 1])

ACS_FOV_ARCSEC = 202.0
ACS_FOV_PIX = 4096.0
pc_per_pix = dist_leoa * 1000.0 * (ACS_FOV_ARCSEC / ACS_FOV_PIX) / 206265.0
kpc_per_pix = pc_per_pix / 1000.0
ap_area_pc = np.pi * (H_AP * pc_per_pix) ** 2
print(f"центр: X = {cx0:.1f}, Y = {cy0:.1f};  апертура {H_AP:.0f} пикс = {H_AP * pc_per_pix:.1f} пк")

_pts_all = np.column_stack([x_la, y_la])
_hv = _pts_all[ConvexHull(_pts_all).vertices]

def _clearance(px, py, hv=_hv):
    """Расстояние до края покрытия (выпуклая оболочка звёзд), пикс."""
    a = hv
    b = np.roll(hv, -1, axis=0)
    ab = b - a
    ab2 = np.sum(ab * ab, axis=1)
    P = np.column_stack([px, py])
    best = np.full(len(P), np.inf)
    for i in range(len(a)):
        t = np.clip(((P - a[i]) @ ab[i]) / ab2[i], 0.0, 1.0)
        proj = a[i] + t[:, None] * ab[i]
        best = np.minimum(best, np.hypot(P[:, 0] - proj[:, 0], P[:, 1] - proj[:, 1]))
    return best

_hist, _edges = np.histogram(y_la, bins=800)
_c = 0.5 * (_edges[:-1] + _edges[1:])
_k = np.array([1.0, 2.0, 3.0, 2.0, 1.0]); _k /= _k.sum()
_sm = np.convolve(_hist.astype(float), _k, mode='same')
_i = int(0.30 * 800) + int(np.argmin(_sm[int(0.30 * 800):int(0.70 * 800)]))
gap_lo, gap_hi = _c[_i] - 45.0, _c[_i] + 45.0
print(f"зазор чипов: y = {gap_lo:.0f} … {gap_hi:.0f} пикс")

angles = 2 * np.pi * np.arange(N_PROBE) / N_PROBE
R_kpc = r_grid * kpc_per_pix

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, lbl in zip(axes, (1, 3)):
    stage = parsec_abbr.get(lbl, f'Stage {lbl}')
    tree = cKDTree(_pts_all[label_leoa == lbl])
    mean = np.full(len(r_grid), np.nan)
    sem = np.full(len(r_grid), np.nan)
    n_used = np.zeros(len(r_grid), dtype=int)
    for j, r in enumerate(r_grid):
        px = cx0 + r * np.cos(angles)
        py = cy0 + r * np.sin(angles)
        ok = _clearance(px, py) >= H_AP
        ok &= (py + H_AP < gap_lo) | (py - H_AP > gap_hi)
        n_used[j] = ok.sum()
        if n_used[j] < 2:
            continue
        n_star = tree.query_ball_point(np.column_stack([px[ok], py[ok]]), H_AP,
                                       return_length=True)
        rho = n_star / ap_area_pc
        mean[j] = rho.mean()
        sem[j] = rho.std(ddof=1) / np.sqrt(n_used[j])

    rho0, sig0 = mean[0], sem[0]
    norm = mean / rho0
    sigma_n = np.sqrt((sem / rho0) ** 2 + (mean * sig0 / rho0 ** 2) ** 2)
    sigma_n[0] = sig0 / rho0
    good = np.isfinite(norm)
    a, b = np.polyfit(R_kpc[good], norm[good], 1)

    yerr = np.vstack([np.minimum(sigma_n, norm), sigma_n])
    ax.errorbar(R_kpc, norm, yerr=yerr, color=color_of.get(lbl, 'C0'),
                lw=1.2, marker='o', ms=4, capsize=2, elinewidth=1)
    ax.plot(R_kpc, a * R_kpc + b, color='k', lw=1.5, ls='--', label=f'a = {a:.4g}')
    ax.set_title(f"{stage}  (N = {int((label_leoa == lbl).sum())})", fontweight='bold')
    ax.set_xlabel("R (кпк)")
    ax.set_ylabel("плотность / первая точка")
    ax.set_xlim(0, bins[-1] * kpc_per_pix)
    ax.set_ylim(bottom=0)
    ax.legend()
    ax.grid(True, alpha=0.3)
    print(f"{stage:4s}  ρ₀ = {rho0:.4g} ± {sig0:.2g} зв./пк²   наклон a = {a:.5g} 1/кпк   "
          f"точек на радиус: {n_used.min()}–{n_used.max()}")

plt.suptitle(f"Leo A — профили по {N_PROBE} точкам на радиус", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Leo A: модель «плато + линейный спад» для профилей по точкам из ячейки выше.
# ρ(R) = P                     при R ≪ R_BREAK,
# ρ(R) = P − s · (R − R_BREAK)   при R ≫ R_BREAK.
# R_BREAK свой для каждой стадии.
# Переход сглажен softplus шириной W_SMOOTH:
# ρ(R) = P − s · W · ln(1 + exp((R − R_BREAK) / W)).
# Подгонка по абсолютным плотностям с весами 1/σ². Затем всё делится на P:
# плато становится единицей, погрешность P переносится на точки.

from scipy.optimize import curve_fit

R_BREAK = {1: 0.10, 3: 0.125}   # кпк, конец плато: MS, RGB
W_SMOOTH = 0.015     # кпк, ширина перехода

def plateau_linear(R, P, s, r_break):
    z = (R - r_break) / W_SMOOTH
    return P - s * W_SMOOTH * np.logaddexp(0.0, z)

R_fine = np.linspace(0, bins[-1] * kpc_per_pix, 400)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, lbl in zip(axes, (1, 3)):
    stage = parsec_abbr.get(lbl, f'Stage {lbl}')
    r_break = R_BREAK[lbl]
    model = lambda R, P, s: plateau_linear(R, P, s, r_break)
    tree = cKDTree(_pts_all[label_leoa == lbl])
    mean = np.full(len(r_grid), np.nan)
    sem = np.full(len(r_grid), np.nan)
    for j, r in enumerate(r_grid):
        px = cx0 + r * np.cos(angles)
        py = cy0 + r * np.sin(angles)
        ok = _clearance(px, py) >= H_AP
        ok &= (py + H_AP < gap_lo) | (py - H_AP > gap_hi)
        if ok.sum() < 2:
            continue
        n_star = tree.query_ball_point(np.column_stack([px[ok], py[ok]]), H_AP,
                                       return_length=True)
        rho = n_star / ap_area_pc
        mean[j] = rho.mean()
        # На малых r апертуры перекрываются: независимых около πr / H_AP.
        n_eff = np.clip(np.pi * r / H_AP, 1.0, ok.sum())
        sig_scatter = rho.std(ddof=1) / np.sqrt(n_eff)
        sig_poisson = np.sqrt(max(n_star.mean(), 1.0)) / ap_area_pc / np.sqrt(n_eff)
        sem[j] = max(sig_scatter, sig_poisson)

    good = np.isfinite(mean) & (sem > 0)
    p0 = [np.nanmean(mean[R_kpc < r_break]), 1.0]
    (P, s), cov = curve_fit(model, R_kpc[good], mean[good], p0=p0,
                            sigma=sem[good], absolute_sigma=True)
    sig_P, sig_s = np.sqrt(np.diag(cov))

    slope = -s / P           # наклон в единицах плато, 1/кпк
    sig_slope = np.sqrt((sig_s / P) ** 2 + (s * sig_P / P ** 2) ** 2
                        - 2 * s / P ** 3 * cov[0, 1])

    norm = mean / P
    sigma_n = np.sqrt((sem / P) ** 2 + (mean * sig_P / P ** 2) ** 2)
    yerr = np.vstack([np.minimum(sigma_n, norm), sigma_n])

    ax.errorbar(R_kpc, norm, yerr=yerr, color=color_of.get(lbl, 'C0'),
                lw=0, elinewidth=1, marker='o', ms=4, capsize=2)
    ax.plot(R_fine, model(R_fine, P, s) / P, color='k', lw=1.5, ls='--',
            label=f'наклон = {slope:.3g} ± {sig_slope:.2g}')
    ax.axhline(1.0, color='0.5', lw=0.8, ls=':')
    ax.axvline(r_break, color='0.5', lw=0.8, ls=':')
    ax.set_title(f"{stage}  (N = {int((label_leoa == lbl).sum())})", fontweight='bold', fontsize=17)
    ax.set_xlabel("R (кпк)", fontsize=16)
    ax.set_ylabel("плотность / плато", fontsize=16)
    ax.tick_params(labelsize=14)
    ax.set_xlim(0, bins[-1] * kpc_per_pix)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=14)
    ax.grid(True, alpha=0.3)
    print(f"{stage:4s}  плато до {r_break:.3f} кпк:  P = {P:.4g} ± {sig_P:.2g} зв./пк²   "
          f"наклон = {slope:.4g} ± {sig_slope:.2g} 1/кпк")

plt.suptitle("Leo A — плато и линейный спад (MS: 0.10 кпк, RGB: 0.125 кпк)", fontsize=18, y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# Leo A: модель «плато + линейный спад» для профилей по точкам из ячейки выше.
# ρ(R) = P                     при R ≪ R_BREAK,
# ρ(R) = P − s · (R − R_BREAK)   при R ≫ R_BREAK.
# R_BREAK свой для каждой стадии.
# Переход сглажен softplus шириной W_SMOOTH:
# ρ(R) = P − s · W · ln(1 + exp((R − R_BREAK) / W)).
# Подгонка по абсолютным плотностям с весами 1/σ². Затем всё делится на P:
# плато становится единицей, погрешность P переносится на точки.

from scipy.optimize import curve_fit

R_BREAK = {1: 0.10, 3: 0.125}   # кпк, конец плато: MS, RGB
W_SMOOTH = 0.015     # кпк, ширина перехода

def plateau_linear(R, P, s, r_break):
    z = (R - r_break) / W_SMOOTH
    return P - s * W_SMOOTH * np.logaddexp(0.0, z)

R_fine = np.linspace(0, bins[-1] * kpc_per_pix, 400)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, lbl in zip(axes, (1, 3)):
    stage = parsec_abbr.get(lbl, f'Stage {lbl}')
    r_break = R_BREAK[lbl]
    model = lambda R, P, s: plateau_linear(R, P, s, r_break)
    tree = cKDTree(_pts_all[label_leoa == lbl])
    mean = np.full(len(r_grid), np.nan)
    sem = np.full(len(r_grid), np.nan)
    for j, r in enumerate(r_grid):
        px = cx0 + r * np.cos(angles)
        py = cy0 + r * np.sin(angles)
        ok = _clearance(px, py) >= H_AP
        ok &= (py + H_AP < gap_lo) | (py - H_AP > gap_hi)
        if ok.sum() < 2:
            continue
        n_star = tree.query_ball_point(np.column_stack([px[ok], py[ok]]), H_AP,
                                       return_length=True)
        rho = n_star / ap_area_pc
        mean[j] = rho.mean()
        # На малых r апертуры перекрываются: независимых около πr / H_AP.
        n_eff = np.clip(np.pi * r / H_AP, 1.0, ok.sum())
        sig_scatter = rho.std(ddof=1) / np.sqrt(n_eff)
        sig_poisson = np.sqrt(max(n_star.mean(), 1.0)) / ap_area_pc / np.sqrt(n_eff)
        sem[j] = max(sig_scatter, sig_poisson)

    good = np.isfinite(mean) & (sem > 0)
    p0 = [np.nanmean(mean[R_kpc < r_break]), 1.0]
    (P, s), cov = curve_fit(model, R_kpc[good], mean[good], p0=p0,
                            sigma=sem[good], absolute_sigma=True)
    sig_P, sig_s = np.sqrt(np.diag(cov))

    slope = -s / P           # наклон в единицах плато, 1/кпк
    sig_slope = np.sqrt((sig_s / P) ** 2 + (s * sig_P / P ** 2) ** 2
                        - 2 * s / P ** 3 * cov[0, 1])

    norm = mean / P
    sigma_n = np.sqrt((sem / P) ** 2 + (mean * sig_P / P ** 2) ** 2)
    yerr = np.vstack([np.minimum(sigma_n, norm), sigma_n])

    ax.errorbar(R_kpc, norm, yerr=yerr, color=color_of.get(lbl, 'C0'),
                lw=0, elinewidth=1, marker='o', ms=4, capsize=2)
    ax.plot(R_fine, model(R_fine, P, s) / P, color='k', lw=1.5, ls='--',
            label=f'наклон = {slope:.3g} ± {sig_slope:.2g}')
    ax.axhline(1.0, color='0.5', lw=0.8, ls=':')
    ax.axvline(r_break, color='0.5', lw=0.8, ls=':')
    ax.set_title(f"{stage}  (N = {int((label_leoa == lbl).sum())})", fontweight='bold', fontsize=17)
    ax.set_xlabel("R (кпк)", fontsize=16)
    ax.set_ylabel("плотность / плато", fontsize=16)
    ax.tick_params(labelsize=14)
    ax.set_xlim(0, bins[-1] * kpc_per_pix)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=14)
    ax.grid(True, alpha=0.3)
    print(f"{stage:4s}  плато до {r_break:.3f} кпк:  P = {P:.4g} ± {sig_P:.2g} зв./пк²   "
          f"наклон = {slope:.4g} ± {sig_slope:.2g} 1/кпк")

plt.suptitle("Leo A — плато и линейный спад (MS: 0.10 кпк, RGB: 0.125 кпк)", fontsize=18, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Leo A: модель «плато + линейный спад» для профилей по точкам из ячейки выше.
# ρ(R) = P                     при R ≪ R_BREAK,
# ρ(R) = P − s · (R − R_BREAK)   при R ≫ R_BREAK.
# R_BREAK свой для каждой стадии.
# Плато строго до R_BREAK. Переход сглажен только справа от излома,
# на отрезке [R_BREAK, R_BREAK + 2W] — парабола, гладко стыкующая плато и прямую.
# Подгонка по абсолютным плотностям с весами 1/σ². Затем всё делится на P:
# плато становится единицей, погрешность P переносится на точки.
# Верхний ряд — ρ, нижний — log10 ρ; погрешность log ρ = σ / (ρ ln 10).
# По log ρ подгоняется своя модель той же формы: плато, затем прямая (наклон в dex/кпк).

from scipy.optimize import curve_fit

R_BREAK = {1: 0.09, 3: 0.125}   # кпк, конец плато: MS, RGB
W_SMOOTH = 0.04     # кпк, ширина перехода

def plateau_linear(R, P, s, r_break):
    d = np.asarray(R, dtype=float) - r_break
    w = W_SMOOTH
    drop = np.where(d <= 0, 0.0,
                    np.where(d < 2 * w, d ** 2 / (4 * w), d - w))
    return P - s * drop

R_fine = np.linspace(0, bins[-1] * kpc_per_pix, 400)

results = {}
for lbl in (1, 3):
    r_break = R_BREAK[lbl]
    model = lambda R, P, s, r_break=r_break: plateau_linear(R, P, s, r_break)
    tree = cKDTree(_pts_all[label_leoa == lbl])
    mean = np.full(len(r_grid), np.nan)
    sem = np.full(len(r_grid), np.nan)
    for j, r in enumerate(r_grid):
        px = cx0 + r * np.cos(angles)
        py = cy0 + r * np.sin(angles)
        ok = _clearance(px, py) >= H_AP
        ok &= (py + H_AP < gap_lo) | (py - H_AP > gap_hi)
        if ok.sum() < 2:
            continue
        n_star = tree.query_ball_point(np.column_stack([px[ok], py[ok]]), H_AP,
                                       return_length=True)
        rho = n_star / ap_area_pc
        mean[j] = rho.mean()
        # Апертуры на малых r перекрываются и меряют одну и ту же центральную область.
        # Консервативно: одна независимая апертура на дугу длиной 2π·H_AP, т.е. r / H_AP.
        n_eff = np.clip(r / H_AP, 1.0, ok.sum())
        sig_scatter = rho.std(ddof=1) / np.sqrt(n_eff)
        sig_poisson = np.sqrt(max(n_star.mean(), 1.0)) / ap_area_pc / np.sqrt(n_eff)
        sem[j] = max(sig_scatter, sig_poisson)

    good = np.isfinite(mean) & (sem > 0)
    p0 = [np.nanmean(mean[R_kpc < r_break]), 1.0]
    (P, s), cov = curve_fit(model, R_kpc[good], mean[good], p0=p0,
                            sigma=sem[good], absolute_sigma=True)
    sig_P, sig_s = np.sqrt(np.diag(cov))

    slope = -s / P           # наклон в единицах плато, 1/кпк
    sig_slope = np.sqrt((sig_s / P) ** 2 + (s * sig_P / P ** 2) ** 2
                        - 2 * s / P ** 3 * cov[0, 1])

    norm = mean / P
    sigma_n = np.sqrt((sem / P) ** 2 + (mean * sig_P / P ** 2) ** 2)
    ok_log = np.isfinite(norm) & (norm > 0) & np.isfinite(sigma_n) & (sigma_n > 0)
    log_y = np.log10(norm[ok_log])
    log_err = sigma_n[ok_log] / (norm[ok_log] * np.log(10))
    (L0, sl), cov_l = curve_fit(model, R_kpc[ok_log], log_y, p0=[0.0, 1.0],
                                sigma=log_err, absolute_sigma=True)
    log_slope = -sl
    sig_log_slope = np.sqrt(cov_l[1, 1])

    results[lbl] = dict(model=model, P=P, s=s, sig_P=sig_P, slope=slope,
                        sig_slope=sig_slope, norm=norm, sigma_n=sigma_n,
                        L0=L0, sl=sl, log_slope=log_slope, sig_log_slope=sig_log_slope)
    print(f"{parsec_abbr.get(lbl, lbl):4s}  плато до {r_break:.3f} кпк:  "
          f"P = {P:.4g} ± {sig_P:.2g} зв./пк²   наклон = {slope:.4g} ± {sig_slope:.2g} 1/кпк   "
          f"по log ρ: {log_slope:.4g} ± {sig_log_slope:.2g} dex/кпк")

fig, axes = plt.subplots(2, 2, figsize=(13, 10.4))
for col, lbl in enumerate((1, 3)):
    res = results[lbl]
    stage = parsec_abbr.get(lbl, f'Stage {lbl}')
    r_break = R_BREAK[lbl]
    curve = res['model'](R_fine, res['P'], res['s']) / res['P']
    for row, logy in enumerate((False, True)):
        ax = axes[row, col]
        norm, sigma_n = res['norm'], res['sigma_n']
        pos = curve > 0
        if logy:
            ok_pt = np.isfinite(norm) & (norm > 0)
            y_pts = np.log10(norm[ok_pt])
            y_err = sigma_n[ok_pt] / (norm[ok_pt] * np.log(10))
            ax.errorbar(R_kpc[ok_pt], y_pts, yerr=y_err,
                        color=color_of.get(lbl, 'C0'),
                        lw=0, elinewidth=1, marker='o', ms=4, capsize=2)
            ax.plot(R_fine, res['model'](R_fine, res['L0'], res['sl']),
                    color='k', lw=1.5, ls='--',
                    label=f"наклон: {res['log_slope']:.3g} ± {res['sig_log_slope']:.2g}")
            ax.axhline(0.0, color='0.5', lw=0.8, ls=':')
            ax.set_ylabel(r"$\log\,\rho$", fontsize=16)
        else:
            lo = np.minimum(sigma_n, norm)
            ax.errorbar(R_kpc, norm, yerr=np.vstack([lo, sigma_n]),
                        color=color_of.get(lbl, 'C0'),
                        lw=0, elinewidth=1, marker='o', ms=4, capsize=2)
            ax.plot(R_fine[pos], curve[pos], color='k', lw=1.5, ls='--',
                    label=f"наклон: {res['slope']:.3g} ± {res['sig_slope']:.2g}")
            ax.axhline(1.0, color='0.5', lw=0.8, ls=':')
            ax.set_ylabel("плотность", fontsize=16)
            ax.set_ylim(bottom=0)
        if row == 0:
            ax.set_title(f"{stage}  (N = {int((label_leoa == lbl).sum())})",
                         fontweight='bold', fontsize=17)
        ax.set_xlabel("R, кпк", fontsize=16)
        ax.tick_params(labelsize=14)
        ax.set_xlim(0, bins[-1] * kpc_per_pix)
        ax.legend(fontsize=14)
        ax.grid(True, alpha=0.3)

plt.suptitle(f"Leo A — плато и линейный спад "
             f"(MS: {R_BREAK[1]:g} кпк, RGB: {R_BREAK[3]:g} кпк)", fontsize=18, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Leo A: только log10 ρ для MS и RGB, профили по точкам (ячейка «по 10 точкам» выше).
# Модель по log ρ: плато, затем прямая. Переход закруглён параболой.
# Закругление начинается в LOG_R_BREAK и заканчивается в LOG_R_BREAK + 2·LOG_W_SMOOTH;
# после этого — прямая. Параметры свои, ячейку выше не трогают.
# Погрешности ρ — как в ячейке выше; log ρ: σ / (ρ ln 10). Нормировка на подогнанное плато.

from scipy.optimize import curve_fit

LOG_R_BREAK = {1: 0.09, 3: 0.125}   # кпк, начало закругления: MS, RGB
LOG_W_SMOOTH = {1: 0.04, 3: 0.04}   # кпк, закругление длится 2·W

def plateau_line(R, L0, s, r_break, w):
    d = np.asarray(R, dtype=float) - r_break
    drop = np.where(d <= 0, 0.0,
                    np.where(d < 2 * w, d ** 2 / (4 * w), d - w))
    return L0 - s * drop

R_fine = np.linspace(0, bins[-1] * kpc_per_pix, 400)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, lbl in zip(axes, (1, 3)):
    stage = parsec_abbr.get(lbl, f'Stage {lbl}')
    r_break, w = LOG_R_BREAK[lbl], LOG_W_SMOOTH[lbl]
    model = lambda R, L0, s, r_break=r_break, w=w: plateau_line(R, L0, s, r_break, w)

    tree = cKDTree(_pts_all[label_leoa == lbl])
    mean = np.full(len(r_grid), np.nan)
    sem = np.full(len(r_grid), np.nan)
    for j, r in enumerate(r_grid):
        px = cx0 + r * np.cos(angles)
        py = cy0 + r * np.sin(angles)
        ok = _clearance(px, py) >= H_AP
        ok &= (py + H_AP < gap_lo) | (py - H_AP > gap_hi)
        if ok.sum() < 2:
            continue
        n_star = tree.query_ball_point(np.column_stack([px[ok], py[ok]]), H_AP,
                                       return_length=True)
        rho = n_star / ap_area_pc
        mean[j] = rho.mean()
        n_eff = np.clip(r / H_AP, 1.0, ok.sum())
        sig_scatter = rho.std(ddof=1) / np.sqrt(n_eff)
        sig_poisson = np.sqrt(max(n_star.mean(), 1.0)) / ap_area_pc / np.sqrt(n_eff)
        sem[j] = max(sig_scatter, sig_poisson)

    ok_pt = np.isfinite(mean) & (mean > 0) & (sem > 0)
    log_y = np.log10(mean[ok_pt])
    log_err = sem[ok_pt] / (mean[ok_pt] * np.log(10))
    p0 = [np.mean(log_y[R_kpc[ok_pt] < r_break]), 1.0]
    (L0, s), cov = curve_fit(model, R_kpc[ok_pt], log_y, p0=p0,
                             sigma=log_err, absolute_sigma=True)
    sig_L0, sig_s = np.sqrt(np.diag(cov))

    # Плато → 0: log(ρ / плато). Погрешность плато добавлена к точкам.
    y_pts = log_y - L0
    y_err = np.sqrt(log_err ** 2 + sig_L0 ** 2)

    ax.errorbar(R_kpc[ok_pt], y_pts, yerr=y_err, color=color_of.get(lbl, 'C0'),
                lw=0, elinewidth=1, marker='o', ms=4, capsize=2)
    ax.plot(R_fine, model(R_fine, L0, s) - L0, color='k', lw=1.5, ls='--',
            label=f"наклон: {-s:.3g} ± {sig_s:.2g}")
    ax.set_title(f"{stage}  (N = {int((label_leoa == lbl).sum())})",
                 fontweight='bold', fontsize=17)
    ax.set_xlabel("R, кпк", fontsize=16)
    ax.set_ylabel(r"$\log\,\rho$", fontsize=16)
    ax.tick_params(labelsize=14)
    ax.set_xlim(0, bins[-1] * kpc_per_pix)
    ax.legend(fontsize=14)
    ax.grid(True, alpha=0.3)
    print(f"{stage:4s}  закругление {r_break:.3f}–{r_break + 2 * w:.3f} кпк:  "
          f"плато log ρ = {L0:.4f} ± {sig_L0:.2g}   наклон = {-s:.4g} ± {sig_s:.2g} dex/кпк")

plt.suptitle(r"Leo A — $\log\,\rho$: плато и линейный спад", fontsize=18, y=1.03)
plt.tight_layout()
plt.show()
